In [1]:

# ============================================================
# CoRDiB SUPERVISOR REVISION
# STEP 01A — ORIGINAL ARTIFACT RECOVERY INVENTORY
#
# READ-ONLY for all existing research files.
# No model fitting, no data transformations, no overwriting.
# Only creates a NEW, uniquely named audit-output folder.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import csv
import hashlib
import json
import os
import re
import uuid

ROOT = Path(r"D:\LUAD_DATAFIRST_METHOD_2026")

CMR = Path(
    r"D:\LUAD_CMR_2026"
    r"\CELL03_DATA_20261021T090947_420335Z_d6aed8ec"
    r"\cmr_gene_expression.npz"
)

HASH_LIMIT_MB = 256

# Exact historical run IDs recovered from exported HTML notebooks.
ARCHIVED = {
    "12_multiresolution_ood_evidence": (
        "20261023T151555Z_638a3c88",
        ["LOCO_PREDICTIONS.csv", "RUN_MANIFEST.json"]
    ),
    "14_quantile_residual_transfer": (
        "20261023T160742Z_f9435239",
        ["LOCO_PREDICTIONS.csv", "RUN_MANIFEST.json"]
    ),
    "15_residual_mechanism_validation": (
        "20261023T162612Z_29518c78",
        ["RUN_MANIFEST.json"]
    ),
    "16_source_referenced_residual": (
        "20261023T170912Z_6ddabbc0",
        [
            "LOCO_PREDICTIONS.csv",
            "RUN_MANIFEST.json"
        ]
    ),
    "17_frozen_reference_inference": (
        "20261023T174703Z_b0b89644",
        [
            "BANK_EVALUATION.csv",
            "INVARIANCE_CHECK.json",
            "RUN_MANIFEST.json"
        ]
    ),
    "18_frozen_method_training": (
        "20261023T175603Z_c0d033f9",
        [
            "FINAL_META_MODEL.npz",
            "FOUR_SOURCE_OOF_META_INPUTS.csv",
            "RUN_MANIFEST.json"
        ]
    ),
    "19_exact_source_base_refit": (
        "20261023T190611Z_a22f33e7",
        [
            "FINAL_lr128.npz",
            "FINAL_lr512.npz",
            "FINAL_META_MODEL.npz",
            "RUN_MANIFEST.json"
        ]
    ),
    "20_integrated_inference_verification": (
        "20261023T191159Z_de09912c",
        ["RUN_MANIFEST.json"]
    ),
    "21_external_gene_eligibility": (
        "20261023T194857Z_9fdc35e5",
        ["RUN_MANIFEST.json"]
    ),
    "22_assay_missingness_bridge_audit": (
        "20261023T195608Z_c617ee1c",
        ["RUN_MANIFEST.json"]
    ),
    "23_external_prediction_lock": (
        "20261023T200343Z_0abb47df",
        [
            "FROZEN_EXTERNAL_PREDICTIONS_UNSCORED.csv",
            "FROZEN_PREDICTION_RECEIPT.json",
            "RUN_MANIFEST.json"
        ]
    ),
    "24_locked_external_label_validation": (
        "20261023T201758Z_8a4f4ab4",
        [
            "SCORED_FROZEN_EXTERNAL_PREDICTIONS.csv",
            "CONFUSION_MATRIX.csv",
            "CLASSWISE_METRICS.csv",
            "STRATIFIED_BOOTSTRAP_95CI.csv",
            "RUN_MANIFEST.json"
        ]
    ),
    "25_external_matched_baseline_audit": (
        "20261024T052011Z_a71b4922",
        ["RUN_MANIFEST.json"]
    ),
    "28_matched_mechanism_comparison": (
        "20261024T062048Z_81cc728e",
        [
            "EQUAL_COHORT_MODEL_METRICS.csv",
            "PAIRED_NLL_COMPARISONS.csv",
            "RUN_MANIFEST.json"
        ]
    )
}

# Additional phases identified in the historical notebooks.
OTHER_PHASES = [
    "26_development_calibrated_external_controls",
    "27_resolution_disagreement_information_audit",
    "29_readonly_reproducibility_audit",
    "29_leakage_integrity_audit",
    "29_mutation_label_provenance_audit",
    "29_manuscript_evidence_consolidation",
    "30_manuscript_figures",
    "32_biological_interpretation",
    "33_submission_display_items",
    "34_submission_quality_assurance",
    "35_final_publication_figures",
    "36_manuscript_assembly"
]

if not ROOT.is_dir():
    raise FileNotFoundError(
        f"Project folder not found: {ROOT}\n"
        "Run this cell in Windows Jupyter with the project "
        "drive accessible. If the project was moved, "
        "update ROOT and CMR above."
    )


# ------------------------------------------------------------
# 1. SHA256 fingerprinting
# ------------------------------------------------------------

def sha256_limited(path):
    if not path.is_file():
        return "NOT_FOUND"

    if path.stat().st_size > HASH_LIMIT_MB * 1024 * 1024:
        return "SKIPPED_GT_256MB"

    hasher = hashlib.sha256()

    with path.open("rb") as handle:
        for block in iter(
            lambda: handle.read(4 * 1024 * 1024),
            b""
        ):
            hasher.update(block)

    return hasher.hexdigest()


def file_record(phase, run_id, filename, path):
    exists = path.is_file()
    size = path.stat().st_size if exists else None

    if not exists:
        status = "MISSING"
    elif size == 0:
        status = "EMPTY"
    else:
        status = "FOUND"

    return {
        "phase": phase,
        "archived_run": run_id,
        "file": filename,
        "status": status,
        "bytes": size,
        "sha256": (
            sha256_limited(path)
            if exists and size
            else None
        ),
        "path": str(path)
    }


# ------------------------------------------------------------
# 2. Verify exact archived runs and required files
# ------------------------------------------------------------

checks = []
phase_status = []

checks.append(
    file_record(
        "MASTER_EXPRESSION",
        "SOURCE",
        CMR.name,
        CMR
    )
)

for phase, (run_id, expected_files) in ARCHIVED.items():

    phase_dir = ROOT / phase
    archived_run = phase_dir / run_id

    alternatives = (
        sorted(
            p.name
            for p in phase_dir.iterdir()
            if p.is_dir()
            and re.match(r"^20\d{6}T", p.name)
        )
        if phase_dir.is_dir()
        else []
    )

    phase_status.append({
        "phase": phase,
        "exact_archived_run_exists": archived_run.is_dir(),
        "other_run_count": len([
            r for r in alternatives
            if r != run_id
        ]),
        "available_run_ids": alternatives
    })

    for filename in expected_files:
        checks.append(
            file_record(
                phase,
                run_id,
                filename,
                archived_run / filename
            )
        )


# ------------------------------------------------------------
# 3. Inspect additional research phase directories
# ------------------------------------------------------------

other_phase_status = []

for phase in OTHER_PHASES:

    path = ROOT / phase

    subdirs = (
        [d for d in path.iterdir() if d.is_dir()]
        if path.is_dir()
        else []
    )

    other_phase_status.append({
        "phase": phase,
        "folder_exists": path.is_dir(),
        "subfolder_count": len(subdirs)
    })


# ------------------------------------------------------------
# 4. Inspect original .ipynb without executing code
# ------------------------------------------------------------

notebook = ROOT / "notebooks" / "00_DATA_FIRST.ipynb"

notebook_info = {
    "path": str(notebook),
    "exists": notebook.is_file(),
    "code_cells": None
}

if notebook.is_file():

    try:
        nb = json.loads(
            notebook.read_text(encoding="utf-8-sig")
        )

        notebook_info["code_cells"] = sum(
            cell.get("cell_type") == "code"
            and bool(
                "".join(
                    cell.get("source", [])
                ).strip()
            )
            for cell in nb.get("cells", [])
        )

    except Exception as exc:
        notebook_info["parse_error"] = repr(exc)


# ------------------------------------------------------------
# 5. Index all project filenames, sizes and modification times
# ------------------------------------------------------------

all_files = []
extension_counts = Counter()

SCAN_LIMIT = 200000
scan_truncated = False

for folder, subfolders, filenames in os.walk(ROOT):

    # Exclude previously generated revision audits.
    subfolders[:] = [
        d for d in subfolders
        if d != "99_CoRDiB_REVISION_AUDIT"
    ]

    for filename in filenames:

        path = Path(folder) / filename

        try:
            stat = path.stat()

            all_files.append({
                "relative_path": str(
                    path.relative_to(ROOT)
                ),
                "bytes": stat.st_size,
                "modified_utc": (
                    datetime.fromtimestamp(
                        stat.st_mtime,
                        timezone.utc
                    ).isoformat()
                )
            })

            extension_counts[
                path.suffix.lower() or "[no extension]"
            ] += 1

        except (OSError, PermissionError):
            pass

        if len(all_files) >= SCAN_LIMIT:
            scan_truncated = True
            break

    if scan_truncated:
        break


# ------------------------------------------------------------
# 6. Save audit in a NEW directory only
# ------------------------------------------------------------

run_tag = (
    datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%SZ"
    )
    + "_"
    + uuid.uuid4().hex[:8]
)

OUT = (
    ROOT
    / "99_CoRDiB_REVISION_AUDIT"
    / ("STEP01A_" + run_tag)
)

OUT.mkdir(parents=True, exist_ok=False)


def save_csv(path, rows, fields):

    with path.open(
        "w",
        newline="",
        encoding="utf-8-sig"
    ) as file:

        writer = csv.DictWriter(
            file,
            fieldnames=fields
        )

        writer.writeheader()
        writer.writerows(rows)


save_csv(
    OUT / "CRITICAL_FILE_CHECKS.csv",
    checks,
    [
        "phase",
        "archived_run",
        "file",
        "status",
        "bytes",
        "sha256",
        "path"
    ]
)

save_csv(
    OUT / "ALL_PROJECT_FILES.csv",
    all_files,
    [
        "relative_path",
        "bytes",
        "modified_utc"
    ]
)


summary = {
    "project_root": str(ROOT),
    "master_expression": str(CMR),
    "audit_run": str(OUT),
    "critical_status_counts": dict(
        Counter(r["status"] for r in checks)
    ),
    "critical_missing": [
        r["path"]
        for r in checks
        if r["status"] != "FOUND"
    ],
    "archived_runs": phase_status,
    "other_phases": other_phase_status,
    "notebook": notebook_info,
    "indexed_files": len(all_files),
    "extension_counts": dict(extension_counts),
    "scan_truncated": scan_truncated,
    "hash_limit_mb": HASH_LIMIT_MB,
    "note": (
        "File presence, size and SHA256 fingerprints only. "
        "This is not a full prediction replay or a verification "
        "against recorded manifest hashes."
    )
}

(OUT / "SUMMARY.json").write_text(
    json.dumps(
        summary,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# 7. Print concise verification report
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("CoRDiB STEP 01A - ORIGINAL ARTIFACT RECOVERY INVENTORY")
print("=" * 72)

print("PROJECT_ROOT:", ROOT)

print(
    "MASTER_EXPRESSION:",
    checks[0]["status"],
    "|",
    CMR
)

print(
    "ARCHIVED_RUNS_FOUND:",
    sum(
        r["exact_archived_run_exists"]
        for r in phase_status
    ),
    "/",
    len(phase_status)
)

print(
    "CRITICAL_FILES:",
    summary["critical_status_counts"]
)

print(
    "OTHER_PHASE_FOLDERS_FOUND:",
    sum(
        r["folder_exists"]
        for r in other_phase_status
    ),
    "/",
    len(other_phase_status)
)

print(
    "INDEXED_FILES:",
    len(all_files),
    "SCAN_TRUNCATED:",
    scan_truncated
)

print("ORIGINAL_NOTEBOOK:", notebook_info)

print(
    "TOP_EXTENSIONS:",
    extension_counts.most_common(12)
)

print("\nARCHIVED_RUN_GAPS:")

for record in phase_status:

    if not record["exact_archived_run_exists"]:

        print(
            " -",
            record["phase"],
            "| alternative run IDs:",
            record["available_run_ids"][:6]
        )


print("\nCRITICAL_FILE_GAPS:")
print(
    "(Missing exact archived files do not prove "
    "the analysis was never performed.)"
)

for record in checks:

    if record["status"] != "FOUND":

        print(
            " -",
            record["status"],
            record["phase"],
            "/",
            record["file"]
        )


print("\nAUDIT_OUTPUT:", OUT)

print(
    "STEP01A_STATUS:",
    (
        "INCOMPLETE_EVIDENCE"
        if summary["critical_missing"]
        else "CRITICAL_FILES_PRESENT"
    )
)

print(
    "NEXT: Review this inventory before any model rerun."
)


CoRDiB STEP 01A - ORIGINAL ARTIFACT RECOVERY INVENTORY
PROJECT_ROOT: D:\LUAD_DATAFIRST_METHOD_2026
MASTER_EXPRESSION: FOUND | D:\LUAD_CMR_2026\CELL03_DATA_20261021T090947_420335Z_d6aed8ec\cmr_gene_expression.npz
ARCHIVED_RUNS_FOUND: 14 / 14
CRITICAL_FILES: {'FOUND': 33}
OTHER_PHASE_FOLDERS_FOUND: 12 / 12
INDEXED_FILES: 557 SCAN_TRUNCATED: False
ORIGINAL_NOTEBOOK: {'path': 'D:\\LUAD_DATAFIRST_METHOD_2026\\notebooks\\00_DATA_FIRST.ipynb', 'exists': True, 'code_cells': 0}
TOP_EXTENSIONS: [('.csv', 179), ('.json', 141), ('.pt', 73), ('.npz', 60), ('.png', 26), ('.md', 22), ('.svg', 22), ('.tiff', 22), ('.soft', 5), ('.txt', 3), ('.gmt', 1), ('.tsv', 1)]

ARCHIVED_RUN_GAPS:

CRITICAL_FILE_GAPS:
(Missing exact archived files do not prove the analysis was never performed.)

AUDIT_OUTPUT: D:\LUAD_DATAFIRST_METHOD_2026\99_CoRDiB_REVISION_AUDIT\STEP01A_20261029T051411Z_efdfe07c
STEP01A_STATUS: CRITICAL_FILES_PRESENT
NEXT: Review this inventory before any model rerun.


In [2]:

# ============================================================
# CoRDiB SUPERVISOR REVISION
# STEP 01B - HISTORICAL MANIFEST INTEGRITY AUDIT
#
# Windows Jupyter | Standard-library only
# READ-ONLY for all original research artifacts.
# NO model training, fitting, or prediction recalculation.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import csv
import hashlib
import json
import re
import uuid
import zipfile

ROOT = Path(r"D:\LUAD_DATAFIRST_METHOD_2026")

STEP01A = (
    ROOT / "99_CoRDiB_REVISION_AUDIT"
    / "STEP01A_20261029T051411Z_efdfe07c"
)

MAX_HASH_MB = 2048
SHA = re.compile(r"^[0-9a-fA-F]{64}$")

# ------------------------------------------------------------
# 1. Load the verified Step 01A inventory
# ------------------------------------------------------------

if not STEP01A.is_dir():
    raise FileNotFoundError(
        f"Step 01A folder not found: {STEP01A}"
    )

previous = json.loads(
    (STEP01A / "SUMMARY.json").read_text(
        encoding="utf-8-sig"
    )
)

if (
    previous.get("scan_truncated")
    or previous.get("critical_missing")
):
    raise RuntimeError(
        "Step 01A inventory was incomplete."
    )

with (
    STEP01A / "CRITICAL_FILE_CHECKS.csv"
).open(encoding="utf-8-sig", newline="") as f:
    critical = list(csv.DictReader(f))

if not critical or any(
    x.get("status") != "FOUND"
    for x in critical
):
    raise RuntimeError(
        "Critical file inventory is incomplete."
    )

run_dirs = sorted({
    Path(x["path"]).parent
    for x in critical
    if x["file"] == "RUN_MANIFEST.json"
})

if len(run_dirs) != 14:
    raise RuntimeError(
        f"Expected 14 archived runs; found {len(run_dirs)}"
    )


# ------------------------------------------------------------
# 2. Independent SHA256 computation
# ------------------------------------------------------------

hash_cache = {}

def digest(path):
    path = Path(path)

    if not path.is_file():
        return None, "MISSING"

    if path.stat().st_size > MAX_HASH_MB * 1024**2:
        return None, "SKIPPED_LARGE"

    if path not in hash_cache:

        h = hashlib.sha256()

        try:
            with path.open("rb") as f:
                for chunk in iter(
                    lambda: f.read(4 * 1024**2),
                    b""
                ):
                    h.update(chunk)

            hash_cache[path] = h.hexdigest()

        except OSError as e:
            return (
                None,
                "READ_ERROR:" + type(e).__name__
            )

    return hash_cache[path], "READABLE"


# ------------------------------------------------------------
# 3. Support heterogeneous archived manifest schemas
# ------------------------------------------------------------

MAP_KEYS = {
    "input_checksums",
    "output_checksums",
    "inputs_sha256",
    "outputs_sha256",
    "input_sha256",
    "output_sha256",
    "input_hashes",
    "output_hashes",
    "outputs",
    "source_sha256"
}

LIST_KEYS = {
    "inputs",
    "outputs",
    "artifacts",
    "input_files",
    "output_files",
    "input_artifacts",
    "output_artifacts",
    "source_sha256"
}


def get_entries(manifest):

    found = []

    if not isinstance(manifest, dict):
        return found

    def add(origin, path, value, size=None):

        if (
            isinstance(path, str)
            and isinstance(value, str)
            and SHA.fullmatch(value)
        ):
            found.append(
                (origin, path, value.lower(), size)
            )

    # Flat filename-to-hash manifest.
    if manifest and all(
        isinstance(k, str)
        and isinstance(v, str)
        and SHA.fullmatch(v)
        for k, v in manifest.items()
    ):
        for p, v in manifest.items():
            add("flat_manifest", p, v)

    for key, obj in manifest.items():

        if key in MAP_KEYS and isinstance(obj, dict):

            for p, v in obj.items():

                if isinstance(v, str):
                    add(key, p, v)

                elif isinstance(v, dict):
                    add(
                        key,
                        v.get("path") or p,
                        v.get("sha256")
                        or v.get("checksum"),
                        v.get("bytes")
                        or v.get("size")
                    )

        if key in LIST_KEYS and isinstance(obj, list):

            for i, v in enumerate(obj):

                if isinstance(v, dict):

                    add(
                        f"{key}[{i}]",
                        v.get("path")
                        or v.get("filename")
                        or v.get("file"),
                        v.get("sha256")
                        or v.get("checksum"),
                        (
                            v["bytes"]
                            if "bytes" in v
                            else v.get("size")
                        )
                    )

    source = manifest.get("source")

    if isinstance(source, dict):
        add(
            "source",
            source.get("path"),
            source.get("sha256")
        )

    return list(dict.fromkeys(found))


def resolve_path(ref, base):

    p = Path(ref)

    if p.is_absolute():
        return p, "ABSOLUTE"

    candidates = [
        base / p,
        ROOT / p
    ]

    matches = list(dict.fromkeys(
        c for c in candidates
        if c.is_file()
    ))

    if len(matches) == 1:
        return matches[0], "RESOLVED"

    if len(matches) > 1:
        return None, "AMBIGUOUS"

    return candidates[0], "UNRESOLVED"


# ------------------------------------------------------------
# 4. Verify every referenced hash in 14 manifests
# ------------------------------------------------------------

results = []
manifest_info = []

master_record = next(
    (
        x for x in critical
        if x["phase"] == "MASTER_EXPRESSION"
    ),
    None
)

for run in run_dirs:

    manifest_path = run / "RUN_MANIFEST.json"

    try:
        manifest = json.loads(
            manifest_path.read_text(
                encoding="utf-8-sig"
            )
        )

    except Exception as exc:

        manifest_info.append({
            "run": str(run),
            "status": "PARSE_ERROR",
            "entries": 0,
            "detail": repr(exc)
        })

        continue

    entries = get_entries(manifest)

    # Verify Phase12 against its original CMR source.
    if (
        run.parent.name
        == "12_multiresolution_ood_evidence"
    ):

        source_sha = manifest.get("source_sha256")

        if (
            master_record
            and isinstance(source_sha, str)
            and SHA.fullmatch(source_sha)
        ):
            entries.append((
                "phase12_master_source",
                master_record["path"],
                source_sha.lower(),
                None
            ))

    manifest_info.append({
        "run": str(run),
        "status": "PARSED",
        "entries": len(entries),
        "detail": ""
    })

    for origin, ref, expected, size in entries:

        p, resolution = resolve_path(ref, run)

        if p is None:

            actual = None
            actual_size = None
            status = "AMBIGUOUS"

        else:

            actual, detail = digest(p)

            actual_size = (
                p.stat().st_size
                if p.is_file()
                else None
            )

            if detail == "MISSING":
                status = "MISSING"

            elif detail != "READABLE":
                status = detail

            elif actual != expected:
                status = "HASH_MISMATCH"

            elif (
                size is not None
                and int(size) != actual_size
            ):
                status = "SIZE_MISMATCH"

            else:
                status = "MATCH"

        results.append({
            "run": str(run.relative_to(ROOT)),
            "claim": origin,
            "reference": ref,
            "resolved_path": str(p) if p else "",
            "resolution": resolution,
            "status": status,
            "expected_sha256": expected,
            "actual_sha256": actual or "",
            "expected_bytes": (
                size if size is not None else ""
            ),
            "actual_bytes": (
                actual_size
                if actual_size is not None
                else ""
            )
        })


# ------------------------------------------------------------
# 5. Recheck every original critical file
# ------------------------------------------------------------

critical_checks = []

for x in critical:

    p = Path(x["path"])
    current_hash, state = digest(p)

    old_hash = x.get("sha256")

    if state == "MISSING":
        result = "MISSING"

    elif state != "READABLE":
        result = state

    elif (
        old_hash
        and SHA.fullmatch(old_hash)
        and current_hash != old_hash.lower()
    ):
        result = "CHANGED_SINCE_01A"

    else:
        result = "READABLE_HASHED"

    critical_checks.append({
        "phase": x["phase"],
        "file": x["file"],
        "status": result,
        "sha256": current_hash or "",
        "path": str(p)
    })


# ------------------------------------------------------------
# 6. Verify frozen external prediction lock
# ------------------------------------------------------------

p19 = next(
    (
        p for p in run_dirs
        if p.parent.name
        == "19_exact_source_base_refit"
    ),
    None
)

p23 = next(
    (
        p for p in run_dirs
        if p.parent.name
        == "23_external_prediction_lock"
    ),
    None
)

p24 = next(
    (
        p for p in run_dirs
        if p.parent.name
        == "24_locked_external_label_validation"
    ),
    None
)

lock_checks = []


def check_lock(label, path, expected):

    actual, state = digest(path)

    if (
        not isinstance(expected, str)
        or not SHA.fullmatch(expected)
    ):
        status = "REFERENCE_UNAVAILABLE"

    elif state != "READABLE":
        status = state

    else:
        status = (
            "MATCH"
            if actual == expected.lower()
            else "HASH_MISMATCH"
        )

    lock_checks.append({
        "check": label,
        "file": str(path),
        "status": status,
        "expected_sha256": str(expected or ""),
        "actual_sha256": actual or ""
    })


if p19 and p23 and p24:

    frozen = (
        p23
        / "FROZEN_EXTERNAL_PREDICTIONS_UNSCORED.csv"
    )

    receipt = json.loads(
        (
            p23 / "FROZEN_PREDICTION_RECEIPT.json"
        ).read_text(encoding="utf-8-sig")
    )

    scored_manifest = json.loads(
        (
            p24 / "RUN_MANIFEST.json"
        ).read_text(encoding="utf-8-sig")
    )

    check_lock(
        "PHASE23_RECEIPT_PREDICTION",
        frozen,
        receipt.get("prediction_file_sha256")
    )

    check_lock(
        "PHASE24_RECORDED_PREDICTION",
        frozen,
        scored_manifest.get(
            "locked_prediction_sha256"
        )
    )

    # Immutable SHA256 recorded in the exported
    # original Phase24 analysis code.
    check_lock(
        "ORIGINAL_PHASE24_LOCK_CONSTANT",
        frozen,
        "8dc5c7eda0a663145116d7481e155ac1877c28392eeb5090c895adb9fcff07f2"
    )

    for fname in (
        "FINAL_lr128.npz",
        "FINAL_lr512.npz"
    ):

        check_lock(
            "PHASE23_RECEIPT_BASE_" + fname,
            p19 / fname,
            receipt.get(
                "model_checksums", {}
            ).get(fname)
        )

    check_lock(
        "PHASE23_RECEIPT_META_MODEL",
        p19 / "FINAL_META_MODEL.npz",
        receipt.get("meta_checksum")
    )

else:

    lock_checks.append({
        "check": "REQUIRED_ARCHIVED_PHASES",
        "file": "",
        "status": "MISSING_PHASE",
        "expected_sha256": "",
        "actual_sha256": ""
    })


# ------------------------------------------------------------
# 7. Non-executing critical file-format checks
# ------------------------------------------------------------

format_checks = []

for x in critical:

    p = Path(x["path"])
    ext = p.suffix.lower()

    if not p.is_file():

        state = "MISSING"
        note = ""

    else:

        try:

            if ext == ".json":

                obj = json.loads(
                    p.read_text(encoding="utf-8-sig")
                )

                state = "PARSE_OK"
                note = type(obj).__name__

            elif ext == ".csv":

                with p.open(
                    encoding="utf-8-sig",
                    newline=""
                ) as f:

                    reader = csv.reader(f)
                    header = next(reader, [])

                state = (
                    "HEADER_OK"
                    if header
                    else "EMPTY_HEADER"
                )

                note = ",".join(header[:10])

            elif ext == ".npz":

                with zipfile.ZipFile(p, "r") as z:

                    members = z.infolist()

                    state = (
                        "ZIP_INDEX_OK"
                        if members
                        else "EMPTY_ZIP"
                    )

                    note = f"members={len(members)}"

            else:

                state = "FORMAT_NOT_TESTED"
                note = ext

        except Exception as exc:

            state = "FORMAT_ERROR"
            note = repr(exc)

    format_checks.append({
        "file": x["file"],
        "status": state,
        "detail": note,
        "path": str(p)
    })


# ------------------------------------------------------------
# 8. Create a NEW independent audit-output directory
# ------------------------------------------------------------

run_tag = (
    datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%SZ"
    )
    + "_"
    + uuid.uuid4().hex[:8]
)

OUT = (
    ROOT
    / "99_CoRDiB_REVISION_AUDIT"
    / ("STEP01B_" + run_tag)
)

OUT.mkdir(parents=True, exist_ok=False)


def write_csv(filename, records, columns):

    with (
        OUT / filename
    ).open(
        "w",
        encoding="utf-8-sig",
        newline=""
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=columns
        )

        writer.writeheader()
        writer.writerows(records)


write_csv(
    "MANIFEST_HASH_VERIFICATION.csv",
    results,
    [
        "run",
        "claim",
        "reference",
        "resolved_path",
        "resolution",
        "status",
        "expected_sha256",
        "actual_sha256",
        "expected_bytes",
        "actual_bytes"
    ]
)

write_csv(
    "CRITICAL_FILE_RECHECK.csv",
    critical_checks,
    [
        "phase",
        "file",
        "status",
        "sha256",
        "path"
    ]
)

write_csv(
    "EXTERNAL_LOCK_CHAIN.csv",
    lock_checks,
    [
        "check",
        "file",
        "status",
        "expected_sha256",
        "actual_sha256"
    ]
)

write_csv(
    "CRITICAL_FORMAT_CHECKS.csv",
    format_checks,
    [
        "file",
        "status",
        "detail",
        "path"
    ]
)


# ------------------------------------------------------------
# 9. Save independent audit summary
# ------------------------------------------------------------

manifest_counts = Counter(
    x["status"] for x in results
)

lock_counts = Counter(
    x["status"] for x in lock_checks
)

format_counts = Counter(
    x["status"] for x in format_checks
)

summary = {
    "manifests": manifest_info,
    "manifest_hash_status": dict(manifest_counts),
    "external_lock_status": dict(lock_counts),
    "format_status": dict(format_counts),
    "critical_recheck_status": dict(
        Counter(
            x["status"]
            for x in critical_checks
        )
    ),
    "expected_manifest_count": 14,
    "verified_manifest_count": sum(
        x["status"] == "PARSED"
        for x in manifest_info
    ),
    "hash_records_checked": len(results),
    "limitations": (
        "Some standalone hash fields without explicit "
        "paths are not covered by manifest-entry checks. "
        "ZIP_INDEX_OK does not imply full array decoding "
        "or CRC validation. This is not model replay."
    ),
    "model_retrained": False
}

(OUT / "SUMMARY.json").write_text(
    json.dumps(
        summary,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# 10. Print final report
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("CoRDiB STEP 01B - HISTORICAL MANIFEST INTEGRITY")
print("=" * 72)

print(
    "MANIFESTS_PARSED:",
    summary["verified_manifest_count"],
    "/14"
)

print(
    "MANIFEST_REFERENCES:",
    len(results),
    dict(manifest_counts)
)

print(
    "CRITICAL_RECHECK:",
    summary["critical_recheck_status"]
)

print(
    "FORMAT_CHECKS:",
    dict(format_counts)
)

print(
    "EXTERNAL_LOCK_CHAIN:",
    dict(lock_counts)
)

print("\nISSUES:")

issues = (
    [
        x for x in results
        if x["status"] != "MATCH"
    ][:20]
    +
    [
        x for x in lock_checks
        if x["status"] != "MATCH"
    ][:10]
)

if not issues:
    print("None detected in the checks executed.")

for x in issues:

    print(
        " -",
        x.get("status"),
        x.get("run", x.get("check")),
        x.get("reference", x.get("file"))
    )

for x in manifest_info:

    if x["status"] != "PARSED":

        print(
            " - MANIFEST_PARSE_ERROR:",
            x["run"],
            x["detail"]
        )

hard_problem = any(
    x["status"] in {
        "HASH_MISMATCH",
        "SIZE_MISMATCH",
        "MISSING",
        "AMBIGUOUS"
    }
    for x in results + lock_checks
)

hard_problem |= any(
    x["status"] not in {
        "PARSE_OK",
        "HEADER_OK",
        "ZIP_INDEX_OK"
    }
    for x in format_checks
)

hard_problem |= any(
    x["status"] != "READABLE_HASHED"
    for x in critical_checks
)

hard_problem |= any(
    x["status"] != "PARSED"
    for x in manifest_info
)

if hard_problem:
    final_status = "REVIEW_REQUIRED"

elif any(
    x["status"] != "MATCH"
    for x in results + lock_checks
):
    final_status = "PARTIAL_VERIFICATION"

else:
    final_status = "CHECKED_HASHES_MATCH"

print("\nSTEP01B_STATUS:", final_status)
print("AUDIT_OUTPUT:", OUT)
print("NO_MODELS_TRAINED: TRUE")
print("ORIGINAL_RESEARCH_FILES_MODIFIED: FALSE")


CoRDiB STEP 01B - HISTORICAL MANIFEST INTEGRITY
MANIFESTS_PARSED: 14 /14
MANIFEST_REFERENCES: 276 {'MATCH': 276}
CRITICAL_RECHECK: {'READABLE_HASHED': 33}
FORMAT_CHECKS: {'ZIP_INDEX_OK': 5, 'HEADER_OK': 12, 'PARSE_OK': 16}
EXTERNAL_LOCK_CHAIN: {'MATCH': 6}

ISSUES:
None detected in the checks executed.

STEP01B_STATUS: CHECKED_HASHES_MATCH
AUDIT_OUTPUT: D:\LUAD_DATAFIRST_METHOD_2026\99_CoRDiB_REVISION_AUDIT\STEP01B_20261029T052202Z_3911ab91
NO_MODELS_TRAINED: TRUE
ORIGINAL_RESEARCH_FILES_MODIFIED: FALSE


In [3]:

# ============================================================
# CoRDiB SUPERVISOR REVISION
# STEP 01C - FROZEN PREDICTION & STATISTICAL REPRODUCTION
#
# READ-ONLY original archives
# NO model fitting, tuning, or prediction changes
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import json
import uuid

import numpy as np
import pandas as pd
from scipy.stats import beta
from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    confusion_matrix,
    precision_recall_fscore_support
)

ROOT = Path(r"D:\LUAD_DATAFIRST_METHOD_2026")

CMR = Path(
    r"D:\LUAD_CMR_2026"
    r"\CELL03_DATA_20261021T090947_420335Z_d6aed8ec"
    r"\cmr_gene_expression.npz"
)

STEP01B = (
    ROOT / "99_CoRDiB_REVISION_AUDIT"
    / "STEP01B_20261029T052202Z_3911ab91"
)

P14 = (
    ROOT / "14_quantile_residual_transfer"
    / "20261023T160742Z_f9435239"
)

P28 = (
    ROOT / "28_matched_mechanism_comparison"
    / "20261024T062048Z_81cc728e"
)

P23 = (
    ROOT / "23_external_prediction_lock"
    / "20261023T200343Z_0abb47df"
)

P24 = (
    ROOT / "24_locked_external_label_validation"
    / "20261023T201758Z_8a4f4ab4"
)

COHORTS = (
    "GSE13213",
    "GSE31210",
    "GSE32863",
    "GSE72094"
)

COUNTS = {
    "GSE13213": (45, 15, 57),
    "GSE31210": (127, 20, 79),
    "GSE32863": (17, 22, 19),
    "GSE72094": (47, 154, 241)
}

MODELS = (
    "q_anchor",
    "q_static",
    "q_residual",
    "CORDIB_INTERACTIONS_OFF",
    "D_ONLY_6",
    "GENERIC_CONDITIONAL_6",
    "AFFINE_STACK_6",
    "ANCHOR_NONLINEAR_6",
    "TEMP_EQUAL_MEAN"
)

COMPARATORS = tuple(
    m for m in MODELS
    if m != "q_residual"
)

PCOLS = ["p_egfr", "p_kras", "p_dwt"]

CLASS_NAMES = [
    "EGFR_MUT",
    "KRAS_MUT",
    "EGFR_KRAS_DOUBLE_WT"
]

NBOOT_DEV = 3000
NBOOT_EXT = 1200
SEED = 20261023
TOL = 2e-8


# ------------------------------------------------------------
# 1. Verify mandatory input files and Step 01B status
# ------------------------------------------------------------

required = [
    CMR,
    STEP01B / "SUMMARY.json",
    P14 / "LOCO_PREDICTIONS.csv",
    P28 / "MODEL_PREDICTIONS.csv",
    P28 / "PER_COHORT_MODEL_METRICS.csv",
    P28 / "EQUAL_COHORT_MODEL_METRICS.csv",
    P28 / "PAIRED_NLL_COMPARISONS.csv",
    P28 / "PER_COHORT_PAIRED_COMPARISONS.csv",
    P23 / "FROZEN_EXTERNAL_PREDICTIONS_UNSCORED.csv",
    P24 / "SCORED_FROZEN_EXTERNAL_PREDICTIONS.csv",
    P24 / "CONFUSION_MATRIX.csv",
    P24 / "CLASSWISE_METRICS.csv",
    P24 / "STRATIFIED_BOOTSTRAP_95CI.csv",
    P24 / "LABEL_PROVENANCE_AUDIT.json"
]

missing = [str(p) for p in required if not p.is_file()]

if missing:
    raise FileNotFoundError(
        "Missing required archived inputs:\n"
        + "\n".join(missing)
    )

previous = json.loads(
    (STEP01B / "SUMMARY.json").read_text(
        encoding="utf-8-sig"
    )
)

if (
    previous.get("verified_manifest_count") != 14
    or previous.get("manifest_hash_status") != {"MATCH": 276}
    or previous.get("external_lock_status") != {"MATCH": 6}
):
    raise RuntimeError(
        "STEP01B verification prerequisite failed."
    )


# ------------------------------------------------------------
# 2. Independent audit utilities
# ------------------------------------------------------------

checks = []


def record(name, passed, detail="", difference=None):

    checks.append({
        "check": name,
        "status": "PASS" if passed else "FAIL",
        "detail": str(detail),
        "max_abs_difference": (
            None if difference is None
            else float(difference)
        )
    })


def valid_probabilities(df, name):

    p = df[PCOLS].apply(
        pd.to_numeric,
        errors="raise"
    ).to_numpy(dtype=float)

    good = (
        np.isfinite(p).all()
        and (p >= 0).all()
        and (p <= 1).all()
        and np.max(
            np.abs(p.sum(axis=1) - 1)
        ) <= 1e-7
    )

    if not good:
        raise ValueError(
            f"{name}: invalid probability matrix."
        )

    return p


def metrics(y, p):

    y = np.asarray(y, dtype=int)
    pred = np.argmax(p, axis=1)

    nll = -np.log(
        np.clip(
            p[np.arange(len(y)), y],
            1e-15,
            1
        )
    ).mean()

    macro_f1 = f1_score(
        y,
        pred,
        labels=[0, 1, 2],
        average="macro",
        zero_division=0
    )

    auc = np.mean([
        roc_auc_score(
            y == k,
            p[:, k]
        )
        for k in range(3)
    ])

    brier = np.mean(
        np.sum(
            (p - np.eye(3)[y]) ** 2,
            axis=1
        )
    )

    return {
        "nll": float(nll),
        "macro_f1": float(macro_f1),
        "macro_auc": float(auc),
        "brier": float(brier)
    }


def compare_records(
    name,
    computed,
    archived,
    keys,
    columns,
    tol=TOL
):

    new = pd.DataFrame(computed).copy()

    old = (
        pd.read_csv(archived)
        if isinstance(archived, Path)
        else pd.DataFrame(archived).copy()
    )

    if (
        any(k not in old for k in keys)
        or any(c not in old for c in columns)
    ):
        record(
            name,
            False,
            "Archived table columns missing."
        )
        return

    if (
        new.duplicated(keys).any()
        or old.duplicated(keys).any()
    ):
        record(
            name,
            False,
            "Duplicate comparison keys."
        )
        return

    merged = new.merge(
        old,
        on=keys,
        how="outer",
        suffixes=("_new", "_old"),
        indicator=True
    )

    aligned = (
        len(merged) == len(new) == len(old)
        and (merged["_merge"] == "both").all()
    )

    if not aligned:
        record(
            name,
            False,
            f"Row/key mismatch: "
            f"new={len(new)}, old={len(old)}"
        )
        return

    differences = []

    for col in columns:

        a = pd.to_numeric(
            merged[f"{col}_new"],
            errors="coerce"
        ).to_numpy(float)

        b = pd.to_numeric(
            merged[f"{col}_old"],
            errors="coerce"
        ).to_numpy(float)

        if not (
            np.isfinite(a).all()
            and np.isfinite(b).all()
        ):
            differences.append(np.inf)

        else:
            differences.append(
                float(np.max(np.abs(a - b)))
            )

    error = max(differences)

    record(
        name,
        error <= tol,
        f"{len(new)} rows; tolerance={tol:.0e}",
        error
    )


# ------------------------------------------------------------
# 3. Validate the master development cohort roster
# ------------------------------------------------------------

with np.load(CMR, allow_pickle=False) as z:

    for key in ("gsm", "cohort", "y"):

        if key not in z.files:
            raise ValueError(
                "Missing CMR field: " + key
            )

    roster = pd.DataFrame({
        "gsm": np.char.upper(
            np.char.strip(
                z["gsm"].astype(str)
            )
        ),
        "cohort_cmr": np.char.strip(
            z["cohort"].astype(str)
        ),
        "label_cmr": np.asarray(
            z["y"],
            dtype=int
        )
    })

if (
    len(roster) != 843
    or roster.gsm.duplicated().any()
):
    raise ValueError(
        "Master expression roster mismatch."
    )


# ------------------------------------------------------------
# 4. Load Phase 14 and Phase 28 predictions
# ------------------------------------------------------------

p28 = pd.read_csv(
    P28 / "MODEL_PREDICTIONS.csv",
    dtype={
        "gsm": str,
        "cohort": str,
        "model": str
    }
)

p14 = pd.read_csv(
    P14 / "LOCO_PREDICTIONS.csv",
    dtype={
        "gsm": str,
        "held_cohort": str,
        "model": str
    }
)

for frame in (p28, p14):

    frame["gsm"] = (
        frame.gsm.astype(str)
        .str.strip()
        .str.upper()
    )

    frame["true_class"] = pd.to_numeric(
        frame["true_class"],
        errors="raise"
    ).astype(int)

valid_probabilities(p28, "Phase28")

if (
    len(p28) != 9 * 843
    or set(p28.model) != set(MODELS)
):
    raise ValueError(
        "Phase28 nine-model prediction roster mismatch."
    )

if p28.duplicated(
    ["model", "cohort", "gsm"]
).any():
    raise ValueError(
        "Duplicate Phase28 predictions."
    )


# ------------------------------------------------------------
# 5. Compare archived predictions against master labels
# ------------------------------------------------------------

joined = p28.merge(
    roster,
    on="gsm",
    how="left",
    validate="many_to_one"
)

identity_ok = (
    joined.cohort_cmr.notna().all()
    and (
        joined.cohort == joined.cohort_cmr
    ).all()
    and (
        joined.true_class == joined.label_cmr
    ).all()
)

record(
    "PHASE28_VS_MASTER_SAMPLE_LABELS",
    identity_ok,
    f"matched={joined.cohort_cmr.notna().sum()}"
)

if not identity_ok:
    raise RuntimeError(
        "Development label/identity inconsistency."
    )

for model in MODELS:

    subset = p28[p28.model == model]

    good = (
        len(subset) == 843
        and set(subset.gsm) == set(roster.gsm)
    )

    record(
        "MODEL_ROSTER_" + model,
        good,
        f"rows={len(subset)}"
    )

    if not good:
        raise ValueError(
            "Incomplete model roster: " + model
        )

for cohort in COHORTS:

    subset = roster[
        roster.cohort_cmr == cohort
    ]

    observed = tuple(
        int(
            (subset.label_cmr == k).sum()
        )
        for k in range(3)
    )

    record(
        "COHORT_CLASS_ROSTER_" + cohort,
        observed == COUNTS[cohort],
        observed
    )


# ------------------------------------------------------------
# 6. Phase14 vs Phase28 frozen CoRDiB replay
# ------------------------------------------------------------

old = (
    p14[p14.model == "q_residual"]
    .rename(columns={"held_cohort": "cohort"})
)

base = p28[
    p28.model == "q_residual"
]

aligned = base.merge(
    old[
        ["cohort", "gsm", "true_class"]
        + PCOLS
    ],
    on=["cohort", "gsm"],
    how="outer",
    validate="one_to_one",
    suffixes=("_28", "_14"),
    indicator=True
)

alignment_ok = (
    len(aligned) == 843
    and (aligned["_merge"] == "both").all()
    and (
        aligned.true_class_28
        == aligned.true_class_14
    ).all()
)

if alignment_ok:

    max_probability_error = max(
        float(
            np.max(
                np.abs(
                    aligned[f"{col}_28"]
                    - aligned[f"{col}_14"]
                )
            )
        )
        for col in PCOLS
    )

else:
    max_probability_error = float("inf")

record(
    "PHASE14_28_ARCHIVED_CORDIB_REPLAY",
    alignment_ok and max_probability_error < 1e-10,
    "Same GSM, cohort and class labels",
    max_probability_error
)


# ------------------------------------------------------------
# 7. Independently recalculate all 36 cohort metrics
# ------------------------------------------------------------

per_cohort = []
by_cohort = {}

for cohort in COHORTS:

    by_cohort[cohort] = {}

    for model in MODELS:

        subset = (
            p28[
                (p28.cohort == cohort)
                & (p28.model == model)
            ]
            .sort_values("gsm")
        )

        y = subset.true_class.to_numpy(
            dtype=int
        )

        p = subset[PCOLS].to_numpy(
            dtype=float
        )

        by_cohort[cohort][model] = {
            "gsm": subset.gsm.to_numpy(str),
            "y": y,
            "p": p
        }

        per_cohort.append({
            "cohort": cohort,
            "model": model,
            "n": len(y),
            **metrics(y, p)
        })

compare_records(
    "PER_COHORT_METRICS",
    per_cohort,
    P28 / "PER_COHORT_MODEL_METRICS.csv",
    ["cohort", "model"],
    [
        "n",
        "nll",
        "macro_f1",
        "macro_auc",
        "brier"
    ]
)

summary = (
    pd.DataFrame(per_cohort)
    .groupby("model", sort=False)[
        [
            "nll",
            "macro_f1",
            "macro_auc",
            "brier"
        ]
    ]
    .mean()
    .reset_index()
)

compare_records(
    "EQUAL_COHORT_METRICS",
    summary,
    P28 / "EQUAL_COHORT_MODEL_METRICS.csv",
    ["model"],
    [
        "nll",
        "macro_f1",
        "macro_auc",
        "brier"
    ]
)


# ------------------------------------------------------------
# 8. Reproduce 3000 paired bootstrap comparisons
# ------------------------------------------------------------

rng = np.random.default_rng(SEED)

paired_rows = []
cohort_gain_rows = []

for comparator in COMPARATORS:

    draws = np.empty(
        (len(COHORTS), NBOOT_DEV),
        dtype=float
    )

    gains = []

    for ci, cohort in enumerate(COHORTS):

        a = by_cohort[cohort]["q_residual"]
        b = by_cohort[cohort][comparator]

        if (
            not np.array_equal(
                a["gsm"], b["gsm"]
            )
            or not np.array_equal(
                a["y"], b["y"]
            )
        ):
            raise RuntimeError(
                "Unpaired prediction rows: "
                + cohort + "/" + comparator
            )

        y = a["y"]
        p0 = a["p"]
        p1 = b["p"]

        # Positive difference favors CoRDiB.
        diff = (
            -np.log(
                np.clip(
                    p1[np.arange(len(y)), y],
                    1e-15,
                    1
                )
            )
            + np.log(
                np.clip(
                    p0[np.arange(len(y)), y],
                    1e-15,
                    1
                )
            )
        )

        gain = float(diff.mean())
        gains.append(gain)

        cohort_gain_rows.append({
            "cohort": cohort,
            "comparator": comparator,
            "n": len(y),
            "nll_gain_cordib": gain,
            "macro_f1_gain_cordib": (
                metrics(y, p0)["macro_f1"]
                - metrics(y, p1)["macro_f1"]
            )
        })

        simulated = np.zeros(
            NBOOT_DEV,
            dtype=float
        )

        for k in range(3):

            group = np.flatnonzero(
                y == k
            )

            if len(group) == 0:
                raise ValueError(
                    "Missing bootstrap class."
                )

            sampled = rng.choice(
                group,
                size=(
                    NBOOT_DEV,
                    len(group)
                ),
                replace=True
            )

            simulated += (
                diff[sampled].sum(axis=1)
            )

        draws[ci] = (
            simulated / len(y)
        )

    lower, upper = np.quantile(
        draws.mean(axis=0),
        [0.025, 0.975]
    )

    jackknife = [
        float(
            np.mean(
                np.delete(gains, i)
            )
        )
        for i in range(4)
    ]

    paired_rows.append({
        "comparator": comparator,
        "mean_equal_cohort_nll_gain": float(
            np.mean(gains)
        ),
        "bootstrap_ci_low": float(lower),
        "bootstrap_ci_high": float(upper),
        "cohorts_gain_positive": int(
            np.sum(np.asarray(gains) > 0)
        ),
        "jackknife_min": min(jackknife),
        "jackknife_max": max(jackknife)
    })

compare_records(
    "PAIRED_COHORT_GAIN",
    cohort_gain_rows,
    P28 / "PER_COHORT_PAIRED_COMPARISONS.csv",
    ["cohort", "comparator"],
    [
        "n",
        "nll_gain_cordib",
        "macro_f1_gain_cordib"
    ]
)

compare_records(
    "PAIRED_BOOTSTRAP_3000",
    paired_rows,
    P28 / "PAIRED_NLL_COMPARISONS.csv",
    ["comparator"],
    [
        "mean_equal_cohort_nll_gain",
        "bootstrap_ci_low",
        "bootstrap_ci_high",
        "cohorts_gain_positive",
        "jackknife_min",
        "jackknife_max"
    ]
)


# ------------------------------------------------------------
# 9. Verify external frozen predictions
# ------------------------------------------------------------

frozen = pd.read_csv(
    P23 / "FROZEN_EXTERNAL_PREDICTIONS_UNSCORED.csv",
    dtype={"gsm": str}
)

scored = pd.read_csv(
    P24 / "SCORED_FROZEN_EXTERNAL_PREDICTIONS.csv",
    dtype={"gsm": str}
)

for frame in (frozen, scored):

    frame["gsm"] = (
        frame.gsm.str.strip().str.upper()
    )

    valid_probabilities(
        frame,
        "GSE66863"
    )

    if (
        frame.gsm.isna().any()
        or frame.gsm.duplicated().any()
    ):
        raise RuntimeError(
            "External GSM integrity error."
        )

record(
    "EXTERNAL_ELIGIBILITY_121_TO_114",
    (
        len(frozen) == 121
        and len(scored) == 114
        and set(scored.gsm).issubset(
            set(frozen.gsm)
        )
    ),
    f"frozen={len(frozen)}, eligible={len(scored)}"
)

joined_external = scored.merge(
    frozen[
        ["gsm", "predicted_class"]
        + PCOLS
    ],
    on="gsm",
    how="left",
    suffixes=("_scored", "_frozen"),
    validate="one_to_one"
)

external_probability_error = float(
    max(
        np.max(
            np.abs(
                joined_external[f"{col}_scored"]
                - joined_external[f"{col}_frozen"]
            )
        )
        for col in PCOLS
    )
)

record(
    "EXTERNAL_FROZEN_PROBABILITIES_UNCHANGED",
    external_probability_error < 1e-12,
    "GSM-aligned eligible predictions",
    external_probability_error
)

record(
    "EXTERNAL_PREDICTED_CLASS_UNCHANGED",
    np.array_equal(
        joined_external.predicted_class_scored.to_numpy(int),
        joined_external.predicted_class_frozen.to_numpy(int)
    )
)


# ------------------------------------------------------------
# 10. Recalculate external confusion matrix
# ------------------------------------------------------------

scored["true_class"] = (
    scored.true_class.astype(int)
)

yext = scored.true_class.to_numpy(
    dtype=int
)

pext = scored[PCOLS].to_numpy(
    dtype=float
)

expected_from_genes = np.select(
    [
        (scored.egfr_status == "MUT")
        & (scored.kras_status == "WT"),

        (scored.egfr_status == "WT")
        & (scored.kras_status == "MUT"),

        (scored.egfr_status == "WT")
        & (scored.kras_status == "WT")
    ],
    [0, 1, 2],
    default=-1
)

record(
    "EXTERNAL_ARCHIVED_DUAL_GENE_LABEL_RULE",
    np.array_equal(
        yext,
        expected_from_genes
    ),
    "Archived parsed statuses verified"
)

pred_ext = pext.argmax(axis=1)

cm = confusion_matrix(
    yext,
    pred_ext,
    labels=[0, 1, 2]
)

old_cm = pd.read_csv(
    P24 / "CONFUSION_MATRIX.csv",
    index_col=0
).to_numpy(dtype=int)

record(
    "EXTERNAL_CONFUSION_MATRIX",
    np.array_equal(cm, old_cm),
    cm.tolist()
)


# ------------------------------------------------------------
# 11. Class-specific exact 95% confidence intervals
# ------------------------------------------------------------

prec, recall, cf1, support = (
    precision_recall_fscore_support(
        yext,
        pred_ext,
        labels=[0, 1, 2],
        zero_division=0
    )
)


def exact_ci(success, total):

    lower = (
        0.0
        if success == 0
        else float(
            beta.ppf(
                0.025,
                success,
                total - success + 1
            )
        )
    )

    upper = (
        1.0
        if success == total
        else float(
            beta.ppf(
                0.975,
                success + 1,
                total - success
            )
        )
    )

    return lower, upper


classwise = []
class_cis = []

for k, name in enumerate(CLASS_NAMES):

    tp = int(cm[k, k])
    fn = int(cm[k, :].sum() - tp)
    fp = int(cm[:, k].sum() - tp)
    tn = int(cm.sum() - tp - fn - fp)

    auc = float(
        roc_auc_score(
            yext == k,
            pext[:, k]
        )
    )

    classwise.append({
        "class": name,
        "precision": prec[k],
        "recall": recall[k],
        "f1": cf1[k],
        "support": support[k],
        "one_vs_rest_auroc": auc
    })

    sens_low, sens_high = exact_ci(
        tp,
        tp + fn
    )

    spec_low, spec_high = exact_ci(
        tn,
        tn + fp
    )

    class_cis.append({
        "class": name,
        "TP": tp,
        "FN": fn,
        "TN": tn,
        "FP": fp,
        "sensitivity": tp / (tp + fn),
        "sensitivity_CI_low": sens_low,
        "sensitivity_CI_high": sens_high,
        "specificity": tn / (tn + fp),
        "specificity_CI_low": spec_low,
        "specificity_CI_high": spec_high
    })

compare_records(
    "EXTERNAL_CLASSWISE_ARCHIVED",
    classwise,
    P24 / "CLASSWISE_METRICS.csv",
    ["class"],
    [
        "precision",
        "recall",
        "f1",
        "support",
        "one_vs_rest_auroc"
    ]
)


# ------------------------------------------------------------
# 12. Verify archived external overall metrics
# ------------------------------------------------------------

ext_metrics = metrics(
    yext,
    pext
)

external_point = {
    "accuracy": float(
        np.mean(pred_ext == yext)
    ),
    "macro_f1": ext_metrics["macro_f1"],
    "macro_recall": float(
        np.mean([
            np.mean(
                pred_ext[yext == k] == k
            )
            for k in range(3)
        ])
    ),
    "nll": ext_metrics["nll"],
    "macro_ovr_auroc": ext_metrics["macro_auc"]
}

archive_report = json.loads(
    (
        P24 / "LABEL_PROVENANCE_AUDIT.json"
    ).read_text(encoding="utf-8-sig")
)

reference_metrics = archive_report.get(
    "metrics",
    {}
)

for key, value in external_point.items():

    archived = reference_metrics.get(
        key
    )

    error = (
        abs(value - float(archived))
        if archived is not None
        else float("inf")
    )

    record(
        "EXTERNAL_METRIC_" + key,
        error <= TOL,
        "Phase24 archived metrics",
        error
    )


# ------------------------------------------------------------
# 13. Replay original 1200 external bootstrap replicates
# ------------------------------------------------------------

rng_ext = np.random.default_rng(SEED)

groups = [
    np.flatnonzero(yext == j)
    for j in range(3)
]

bootstrap_rows = []

for _ in range(NBOOT_EXT):

    sample = np.concatenate([
        rng_ext.choice(
            group,
            size=len(group),
            replace=True
        )
        for group in groups
    ])

    yy = yext[sample]
    pp = pext[sample]

    mm = metrics(
        yy,
        pp
    )

    guess = pp.argmax(axis=1)

    bootstrap_rows.append({
        "accuracy": float(
            np.mean(guess == yy)
        ),
        "macro_f1": mm["macro_f1"],
        "macro_recall": float(
            np.mean([
                np.mean(
                    guess[yy == k] == k
                )
                for k in range(3)
            ])
        ),
        "nll": mm["nll"],
        "macro_ovr_auroc": mm["macro_auc"]
    })

boot = pd.DataFrame(
    bootstrap_rows
)

external_boot_ci = []

for metric in (
    "accuracy",
    "macro_f1",
    "macro_recall",
    "nll",
    "macro_ovr_auroc"
):

    lower, upper = np.quantile(
        boot[metric].to_numpy(float),
        [0.025, 0.975]
    )

    external_boot_ci.append({
        "metric": metric,
        "point": external_point[metric],
        "ci95_lower": float(lower),
        "ci95_upper": float(upper)
    })

compare_records(
    "EXTERNAL_ORIGINAL_BOOTSTRAP_1200",
    external_boot_ci,
    P24 / "STRATIFIED_BOOTSTRAP_95CI.csv",
    ["metric"],
    [
        "point",
        "ci95_lower",
        "ci95_upper"
    ]
)


# ------------------------------------------------------------
# 14. Save independent verification outputs
# ------------------------------------------------------------

tag = (
    datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%SZ"
    )
    + "_"
    + uuid.uuid4().hex[:8]
)

OUT = (
    ROOT
    / "99_CoRDiB_REVISION_AUDIT"
    / ("STEP01C_" + tag)
)

OUT.mkdir(
    parents=True,
    exist_ok=False
)

pd.DataFrame(checks).to_csv(
    OUT / "AUDIT_CHECKS.csv",
    index=False
)

pd.DataFrame(per_cohort).to_csv(
    OUT / "RECALCULATED_36_COHORT_METRICS.csv",
    index=False
)

summary.to_csv(
    OUT / "RECALCULATED_9_EQUAL_COHORT_METRICS.csv",
    index=False
)

pd.DataFrame(cohort_gain_rows).to_csv(
    OUT / "RECALCULATED_32_PAIRED_COHORT_GAINS.csv",
    index=False
)

pd.DataFrame(paired_rows).to_csv(
    OUT / "RECALCULATED_8_PAIRED_CIS.csv",
    index=False
)

pd.DataFrame(class_cis).to_csv(
    OUT / "EXTERNAL_EXACT_CLASSWISE_95CI.csv",
    index=False
)

pd.DataFrame(external_boot_ci).to_csv(
    OUT / "RECALCULATED_EXTERNAL_BOOTSTRAP_CIS.csv",
    index=False
)


# ------------------------------------------------------------
# 15. Final audit status
# ------------------------------------------------------------

failures = [
    x for x in checks
    if x["status"] == "FAIL"
]

status = (
    "PASS_VERIFIED_FROZEN_STATISTICS"
    if not failures
    else "REVIEW_REQUIRED"
)

report = {
    "status": status,
    "audit_counts": dict(
        Counter(
            x["status"]
            for x in checks
        )
    ),
    "issues": failures,
    "archive_inputs_read_only": True,
    "model_retraining": False,
    "development_interpretation": (
        "Historically exposed exploratory LOCO evidence"
    ),
    "external_interpretation": (
        "Post-exposure frozen-prediction assessment"
    ),
    "bootstrap_limitation": (
        "Conditional within-observed-cohort uncertainty; "
        "no model-refitting or new-domain uncertainty"
    ),
    "classwise_ci": (
        "Two-sided exact Clopper-Pearson 95% intervals; "
        "descriptive, without multiplicity adjustment"
    ),
    "audit_output": str(OUT)
}

(OUT / "SUMMARY.json").write_text(
    json.dumps(
        report,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# 16. Print complete verification summary
# ------------------------------------------------------------

print("\n" + "=" * 72)
print(
    "STEP 01C - FROZEN PREDICTION "
    "& STATISTICAL REPRODUCTION"
)
print("=" * 72)

print(
    "PHASE14_28_MAX_PROBABILITY_DIFFERENCE:",
    f"{max_probability_error:.3e}"
)

print(
    "COHORT_METRICS:",
    len(per_cohort)
)

print(
    "MODEL_CONFIGURATIONS:",
    len(summary)
)

print(
    "PAIRED_COMPARATORS:",
    len(paired_rows)
)

print("\nCORDIB_EQUAL_COHORT_METRICS:")

print(
    summary[
        summary.model == "q_residual"
    ].round(6).to_string(index=False)
)

print("\nSELECTED_PAIRED_COMPARISONS:")

print(
    pd.DataFrame(paired_rows)
    .set_index("comparator")
    .loc[
        ["D_ONLY_6", "AFFINE_STACK_6"],
        [
            "mean_equal_cohort_nll_gain",
            "bootstrap_ci_low",
            "bootstrap_ci_high"
        ]
    ]
    .round(6)
    .to_string()
)

print(
    "\nEXTERNAL_N:",
    len(frozen),
    "ELIGIBLE:",
    len(scored)
)

print(
    "EXTERNAL_METRICS:",
    {
        k: round(v, 6)
        for k, v in external_point.items()
    }
)

print(
    "\nEXTERNAL_CONFUSION_MATRIX:\n",
    cm
)

print(
    "\nEXTERNAL_CLASSWISE_EXACT_95CI:\n",
    pd.DataFrame(class_cis)
    .round(6)
    .to_string(index=False)
)

print(
    "\nCHECKS:",
    dict(
        Counter(
            x["status"]
            for x in checks
        )
    )
)

if failures:

    print("\nFAILED_CHECKS:")

    for x in failures:

        print(
            " -",
            x["check"],
            "|",
            x["detail"],
            "| difference:",
            x["max_abs_difference"]
        )

else:
    print("\nFAILED_CHECKS: NONE")

print("\nSTEP01C_STATUS:", status)
print("OUTPUT:", OUT)
print("RETRAINED: NO")
print("ORIGINAL_RESEARCH_FILES_MODIFIED: NO")


STEP 01C - FROZEN PREDICTION & STATISTICAL REPRODUCTION
PHASE14_28_MAX_PROBABILITY_DIFFERENCE: 0.000e+00
COHORT_METRICS: 36
MODEL_CONFIGURATIONS: 9
PAIRED_COMPARATORS: 8

CORDIB_EQUAL_COHORT_METRICS:
     model      nll  macro_f1  macro_auc   brier
q_residual 0.909271  0.545245   0.810636 0.55901

SELECTED_PAIRED_COMPARISONS:
                mean_equal_cohort_nll_gain  bootstrap_ci_low  bootstrap_ci_high
comparator                                                                     
D_ONLY_6                          0.005782          0.003233           0.008298
AFFINE_STACK_6                    0.001703         -0.006262           0.009568

EXTERNAL_N: 121 ELIGIBLE: 114
EXTERNAL_METRICS: {'accuracy': 0.666667, 'macro_f1': 0.65548, 'macro_recall': 0.717351, 'nll': 0.796132, 'macro_ovr_auroc': 0.865747}

EXTERNAL_CONFUSION_MATRIX:
 [[16  0  1]
 [ 4 21 14]
 [17  2 39]]

EXTERNAL_CLASSWISE_EXACT_95CI:
               class  TP  FN  TN  FP  sensitivity  sensitivity_CI_low  sensitivity_CI_hi

In [4]:

# ============================================================
# CoRDiB SUPERVISOR REVISION
# STEP 01D - NESTED LOCO / LEAKAGE EVIDENCE AUDIT
# Windows Jupyter | independent READ-ONLY verification
# NO model fitting, retraining or hyperparameter tuning
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import hashlib
import json
import uuid

import numpy as np
import pandas as pd
from scipy.special import expit
from scipy.stats import rankdata

ROOT = Path(r"D:\LUAD_DATAFIRST_METHOD_2026")

CMR = Path(
    r"D:\LUAD_CMR_2026"
    r"\CELL03_DATA_20261021T090947_420335Z_d6aed8ec"
    r"\cmr_gene_expression.npz"
)

P12 = (
    ROOT / "12_multiresolution_ood_evidence"
    / "20261023T151555Z_638a3c88"
)

P14 = (
    ROOT / "14_quantile_residual_transfer"
    / "20261023T160742Z_f9435239"
)

P29_SOURCE = (
    ROOT / "29_leakage_integrity_audit"
    / "29C_source_partition_audit_20261024T182200Z_0a2be2fe"
)

P29_INNER = (
    ROOT / "29_leakage_integrity_audit"
    / "29C_inner_OOF_independent_replay_20261024T190313Z_476bfd42"
)

STEP01C = (
    ROOT / "99_CoRDiB_REVISION_AUDIT"
    / "STEP01C_20261029T053016Z_eff9d634"
)

COHORTS = (
    "GSE13213",
    "GSE31210",
    "GSE32863",
    "GSE72094"
)

COUNTS = {
    "GSE13213": 117,
    "GSE31210": 226,
    "GSE32863": 58,
    "GSE72094": 442
}

PCOLS = ["p_egfr", "p_kras", "p_dwt"]


# ------------------------------------------------------------
# 1. Check required historical files
# ------------------------------------------------------------

needed = [
    CMR,
    P12 / "PROTOCOL.json",
    P12 / "LOCO_PREDICTIONS.csv",
    P14 / "LOCO_PREDICTIONS.csv",
    P14 / "QUANTILE_SCORE_DIAGNOSTICS.csv",
    P29_SOURCE / "AUDIT_CHECKS.csv",
    P29_SOURCE / "CHECKPOINT_REPLAY.csv",
    P29_SOURCE / "AUDIT_MANIFEST.json",
    P29_SOURCE / "OUTPUT_SHA256.json",
    P29_INNER / "INNER_OOF_CHANNEL_REPLAY.csv",
    P29_INNER / "INNER_OOF_PARTITIONS.csv",
    P29_INNER / "RUN_MANIFEST.json",
    P29_INNER / "OUTPUT_SHA256.json",
    STEP01C / "SUMMARY.json"
]

needed += [
    P12 / "nested_oof" / f"{c}.npz"
    for c in COHORTS
]

needed += [
    P14 / "inner_oof" / f"{c}.npz"
    for c in COHORTS
]

needed += [
    P14 / "checkpoints" / f"{c}_q_residual.npz"
    for c in COHORTS
]

missing = [
    str(p)
    for p in needed
    if not p.is_file()
]

if missing:
    raise FileNotFoundError(
        "Missing original archived evidence:\n"
        + "\n".join(missing)
    )

with (
    STEP01C / "SUMMARY.json"
).open(encoding="utf-8-sig") as f:
    previous = json.load(f)

if previous.get("status") != "PASS_VERIFIED_FROZEN_STATISTICS":
    raise RuntimeError(
        "Step 01C has not passed; do not continue."
    )


# ------------------------------------------------------------
# 2. Reusable verification utilities
# ------------------------------------------------------------

def sha256(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as stream:
        for block in iter(
            lambda: stream.read(4 * 1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


def load_json(path):

    return json.loads(
        path.read_text(encoding="utf-8-sig")
    )


checks = []


def audit(name, ok, detail, max_error=None):

    checks.append({
        "check": name,
        "status": (
            "PASS" if bool(ok) else "FAIL"
        ),
        "detail": str(detail),
        "max_abs_error": (
            float(max_error)
            if max_error is not None
            else None
        )
    })


# ------------------------------------------------------------
# 3. Verify original Phase29C audit output hashes
# ------------------------------------------------------------

for folder, label in [
    (P29_SOURCE, "SOURCE"),
    (P29_INNER, "INNER")
]:

    registry = load_json(
        folder / "OUTPUT_SHA256.json"
    )

    matching = bool(registry)
    mismatched = []

    for filename, digest in registry.items():

        path = folder / filename

        good = (
            path.is_file()
            and sha256(path) == digest
        )

        matching &= good

        if not good:
            mismatched.append(filename)

    audit(
        f"PHASE29C_{label}_OUTPUT_HASH_REGISTRY",
        matching,
        (
            f"verified={len(registry)-len(mismatched)}"
            f"/{len(registry)}, bad={mismatched}"
        )
    )


# ------------------------------------------------------------
# 4. Load archived source-partition audits
# ------------------------------------------------------------

src_checks = pd.read_csv(
    P29_SOURCE / "AUDIT_CHECKS.csv"
)

src_state = pd.read_csv(
    P29_SOURCE / "CHECKPOINT_REPLAY.csv"
)

inner_replay = pd.read_csv(
    P29_INNER / "INNER_OOF_CHANNEL_REPLAY.csv"
)

inner_parts = pd.read_csv(
    P29_INNER / "INNER_OOF_PARTITIONS.csv"
)

src_manifest = load_json(
    P29_SOURCE / "AUDIT_MANIFEST.json"
)

inner_manifest = load_json(
    P29_INNER / "RUN_MANIFEST.json"
)

protocol = load_json(
    P12 / "PROTOCOL.json"
)

src_counts = Counter(
    src_checks["status"].astype(str)
)

inner_counts = Counter(
    inner_replay["status"].astype(str)
)

audit(
    "ARCHIVED_29C_SOURCE_STATUS",
    (
        len(src_checks) == 49
        and src_counts == {
            "PASS": 45,
            "WARNING": 4
        }
        and dict(src_counts)
        == src_manifest.get("status_counts")
    ),
    dict(src_counts)
)

audit(
    "ARCHIVED_29C_INNER_STATUS",
    (
        len(inner_replay) == 48
        and inner_counts == {"PASS": 48}
        and dict(inner_counts)
        == inner_manifest.get("check_counts")
    ),
    dict(inner_counts)
)


# ------------------------------------------------------------
# 5. Recheck eight archived outer checkpoint audits
# ------------------------------------------------------------

selected_ok = (
    len(src_state) == 8
    and set(
        zip(
            src_state["held"],
            src_state["resolution"].astype(int)
        )
    ) == {
        (c, r)
        for c in COHORTS
        for r in (128, 512)
    }
    and src_state["feature_match"]
    .astype(str)
    .str.lower()
    .eq("true")
    .all()
    and np.isfinite(
        src_state[
            [
                "max_mu_error",
                "max_sd_error",
                "raw_probability_error"
            ]
        ].to_numpy(float)
    ).all()
    and (
        src_state["max_mu_error"] <= 2e-5
    ).all()
    and (
        src_state["max_sd_error"] <= 2e-5
    ).all()
    and (
        src_state["raw_probability_error"] <= 1e-4
    ).all()
)

audit(
    "ARCHIVED_8_TRAIN_ONLY_CHECKPOINT_REPLAYS",
    selected_ok,
    (
        "8 checkpoints with exact feature selections, "
        "train-only scaling, and outer prediction replay"
    ),
    src_state["raw_probability_error"].max()
)


# ------------------------------------------------------------
# 6. Verify master development roster
# ------------------------------------------------------------

with np.load(
    CMR,
    allow_pickle=False
) as z:

    cohort = np.char.upper(
        np.char.strip(
            z["cohort"].astype(str)
        )
    )

    gsm = np.char.upper(
        np.char.strip(
            z["gsm"].astype(str)
        )
    )

    y = np.asarray(
        z["y"],
        dtype=int
    )

roster = {
    (str(s), str(g)): int(k)
    for s, g, k in zip(
        cohort,
        gsm,
        y
    )
}

roster_ok = (
    len(y) == 843
    and len(roster) == 843
    and {
        c: int((cohort == c).sum())
        for c in COHORTS
    } == COUNTS
)

audit(
    "CMR_DEVELOPMENT_ROSTER",
    roster_ok,
    f"unique cohort/GSM={len(roster)}"
)

if not roster_ok:
    raise RuntimeError(
        "CMR roster changed; no downstream validity assumed."
    )


# ------------------------------------------------------------
# 7. Validate all 12 inner source-held-out partitions
# ------------------------------------------------------------

expected_partitions = {
    (outer, inner): tuple(
        sorted(
            set(COHORTS) - {outer, inner}
        )
    )
    for outer in COHORTS
    for inner in COHORTS
    if inner != outer
}

part_ok = (
    len(inner_parts) == 12
    and not inner_parts.duplicated(
        ["outer_held", "inner_held"]
    ).any()
)

part_details = []

for _, r in inner_parts.iterrows():

    outer = str(r["outer_held"])
    inner = str(r["inner_held"])

    fit_sources = tuple(
        sorted(
            x.strip()
            for x in str(
                r["fitting_sources"]
            ).split(";")
            if x.strip()
        )
    )

    expected = expected_partitions.get(
        (outer, inner)
    )

    good = (
        expected is not None
        and fit_sources == expected
        and int(r["fit_n"]) == sum(
            COUNTS[c]
            for c in fit_sources
        )
        and int(r["prediction_n"])
        == COUNTS.get(inner, -1)
        and int(r["outer_labels_in_fit"]) == 0
        and int(r["inner_labels_in_fit"]) == 0
    )

    part_ok &= good

    part_details.append({
        "outer_held": outer,
        "inner_held": inner,
        "training_sources": ";".join(fit_sources),
        "fit_n": int(r["fit_n"]),
        "pred_n": int(r["prediction_n"]),
        "partition_valid": bool(good)
    })

audit(
    "ALL_12_TWO_SOURCE_FIT_PARTITIONS",
    part_ok,
    (
        "Every inner fitting set excludes both the "
        "outer cohort and inner target cohort"
    )
)


# ------------------------------------------------------------
# 8. Verify archived 48 OOF channel reconstructions
# ------------------------------------------------------------

replay_pairs = set()
replay_ok = len(inner_replay) == 48

for _, r in inner_replay.iterrows():

    outer = str(r["outer_held"])
    inner = str(r["inner_held"])

    key = (
        outer,
        inner,
        int(r["resolution"]),
        str(r["contrast"])
    )

    replay_pairs.add(key)

    good = (
        (outer, inner) in expected_partitions
        and int(r["resolution"]) in (128, 512)
        and r["contrast"] in ("EGFR", "KRAS")
        and int(r["training_n"]) == sum(
            COUNTS[c]
            for c in expected_partitions[
                (outer, inner)
            ]
        )
        and int(r["prediction_n"])
        == COUNTS[inner]
        and r["status"] == "PASS"
        and int(r["convergence_warnings"]) == 0
        and np.isfinite(
            float(r["max_abs_logit_error"])
        )
        and float(r["max_abs_logit_error"]) <= 1e-3
    )

    replay_ok &= good

replay_ok &= (
    len(replay_pairs) == 48
)

audit(
    "ALL_48_INNER_LOGIT_REPLAYS",
    replay_ok,
    (
        "48 unique combinations; all original fitting "
        "replications within tolerance; no new refitting"
    ),
    inner_replay["max_abs_logit_error"].max()
)


# ------------------------------------------------------------
# 9. Independently reconstruct inner OOF source quantiles
# ------------------------------------------------------------

inner_rows = []

for held in COHORTS:

    with np.load(
        P12 / "nested_oof" / f"{held}.npz",
        allow_pickle=False
    ) as z:

        source = np.char.upper(
            np.char.strip(
                z["cohort"].astype(str)
            )
        )

        sample = np.char.upper(
            np.char.strip(
                z["gsm"].astype(str)
            )
        )

        labels = np.asarray(
            z["label"],
            dtype=int
        )

        scores = np.asarray(
            z["oof_scores"],
            dtype=float
        )

    with np.load(
        P14 / "inner_oof" / f"{held}.npz",
        allow_pickle=False
    ) as z:

        source14 = np.char.upper(
            np.char.strip(
                z["source"].astype(str)
            )
        )

        label14 = np.asarray(
            z["label"],
            dtype=int
        )

        scores14 = np.asarray(
            z["oof_scores"],
            dtype=float
        )

        archived_quantiles = np.asarray(
            z["rank_evidence"],
            dtype=float
        )

    rows_expected = 843 - COUNTS[held]

    keys = list(
        zip(
            source.tolist(),
            sample.tolist()
        )
    )

    identity_valid = (
        len(keys) == rows_expected
        and len(set(keys)) == rows_expected
        and held not in set(source)
        and set(source)
        == set(COHORTS) - {held}
        and all(
            roster.get(key) == int(lbl)
            for key, lbl in zip(
                keys,
                labels
            )
        )
    )

    audit(
        f"OUTER_{held}_OOF_SOURCE_EXCLUSION",
        identity_valid,
        (
            f"OOF training rows={len(keys)}; "
            "held-out source excluded"
        )
    )

    reuse_valid = (
        scores.shape == (rows_expected, 4)
        and scores14.shape == scores.shape
        and archived_quantiles.shape == scores.shape
        and np.array_equal(
            source,
            source14
        )
        and np.array_equal(
            labels,
            label14
        )
        and np.array_equal(
            scores,
            scores14
        )
    )

    audit(
        f"OUTER_{held}_PHASE12_14_OOF_LINKAGE",
        reuse_valid,
        (
            "Same source/labels/scores and "
            "four evidence channels"
        )
    )

    if not (
        identity_valid
        and reuse_valid
    ):
        continue

    calculated = np.empty_like(scores)

    for src in np.unique(source):

        idx = np.flatnonzero(
            source == src
        )

        for j in range(4):

            u = np.clip(
                (
                    rankdata(
                        scores[idx, j],
                        method="average"
                    ) - 0.5
                ) / len(idx),
                1e-5,
                1 - 1e-5
            )

            calculated[idx, j] = (
                np.log(u) - np.log1p(-u)
            ) / 2.5

    err = float(
        np.max(
            np.abs(
                calculated
                - archived_quantiles
            )
        )
    )

    audit(
        f"OUTER_{held}_SOURCE_QUANTILE_REPLAY",
        err < 1e-10,
        (
            "Quantiles recomputed within each "
            "inner held-out source without using "
            "mutation labels"
        ),
        err
    )

    inner_rows.append({
        "outer_held": held,
        "OOF_n": rows_expected,
        "source_quantile_max_error": err
    })


# ------------------------------------------------------------
# 10. Reconstruct outer target-batch quantile evidence
# ------------------------------------------------------------

p12 = pd.read_csv(
    P12 / "LOCO_PREDICTIONS.csv",
    dtype={"gsm": str}
)

p14 = pd.read_csv(
    P14 / "LOCO_PREDICTIONS.csv",
    dtype={"gsm": str}
)

qdiag = pd.read_csv(
    P14 / "QUANTILE_SCORE_DIAGNOSTICS.csv"
)

for frame in (p12, p14):

    frame["gsm"] = (
        frame["gsm"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    frame["held_cohort"] = (
        frame["held_cohort"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

outer_rows = []

for held in COHORTS:

    raw = {}

    for resolution in (128, 512):

        subset = p12[
            (p12["held_cohort"] == held)
            & (
                p12["model"]
                == f"raw{resolution}"
            )
        ].sort_values("gsm")

        if (
            len(subset) != COUNTS[held]
            or subset.gsm.duplicated().any()
        ):
            raise ValueError(
                f"Missing/duplicated raw{resolution}: {held}"
            )

        raw[resolution] = subset

    a = raw[128]
    b = raw[512]

    keys_ok = (
        np.array_equal(
            a.gsm.to_numpy(),
            b.gsm.to_numpy()
        )
        and np.array_equal(
            a.true_class.to_numpy(),
            b.true_class.to_numpy()
        )
        and set(a.gsm) == {
            g
            for c, g in roster
            if c == held
        }
        and all(
            roster[(held, str(g))] == int(t)
            for g, t in zip(
                a.gsm,
                a.true_class
            )
        )
    )

    audit(
        f"OUTER_{held}_P12_RAW_MODEL_ALIGNMENT",
        keys_ok,
        (
            f"n={len(a)}, raw128/raw512 "
            "predictions correctly paired"
        )
    )

    if not keys_ok:
        continue

    evidence = []

    for resolution in (128, 512):

        p = raw[resolution][PCOLS].to_numpy(
            float
        )

        if (
            not np.isfinite(p).all()
            or np.max(
                np.abs(
                    p.sum(axis=1) - 1
                )
            ) > 1e-7
        ):
            raise ValueError(
                f"Invalid raw probability simplex in {held}"
            )

        eps = 1e-12

        e = np.clip(
            p[:, 0],
            eps,
            1 - eps
        )

        k = np.clip(
            p[:, 1],
            eps,
            1
        )

        w = np.clip(
            p[:, 2],
            eps,
            1
        )

        evidence.append(
            np.column_stack((
                np.log(e / (1 - e)),
                np.log(k / w)
            ))
        )

    logits = np.column_stack((
        evidence[0][:, 0],
        evidence[1][:, 0],
        evidence[0][:, 1],
        evidence[1][:, 1]
    ))

    q = np.empty_like(logits)

    for j in range(4):

        u = np.clip(
            (
                rankdata(
                    logits[:, j],
                    method="average"
                ) - 0.5
            ) / len(logits),
            1e-5,
            1 - 1e-5
        )

        q[:, j] = (
            np.log(u) - np.log1p(-u)
        ) / 2.5


    # --------------------------------------------------------
    # 11. Replay frozen q_residual checkpoint predictions
    #     No optimizer or training step is used here.
    # --------------------------------------------------------

    checkpoint = (
        P14 / "checkpoints"
        / f"{held}_q_residual.npz"
    )

    with np.load(
        checkpoint,
        allow_pickle=False
    ) as z:

        bias = np.asarray(
            z["bias"],
            float
        ).reshape(-1)

        scale = np.asarray(
            z["raw_scale"],
            float
        ).reshape(-1)

        weights = np.asarray(
            z["weights"],
            float
        )

        bound = float(
            np.asarray(
                z["correction_bound"]
            ).reshape(-1)[0]
        )

        gamma = float(
            np.asarray(
                z["guard_lambda"]
            ).reshape(-1)[0]
        )

    if (
        bias.shape != (2,)
        or scale.shape != (2,)
        or weights.shape != (2, 3)
    ):
        raise ValueError(
            f"Invalid residual checkpoint: {held}"
        )

    anchor = q[:, [0, 2]]

    delta = (
        q[:, [1, 3]] - anchor
    )

    features = np.stack(
        [
            delta,
            delta * np.tanh(
                1.5 * anchor
            ),
            np.tanh(2 * delta)
            * (
                1 - np.tanh(anchor) ** 2
            )
        ],
        axis=-1
    )

    correction = np.einsum(
        "nhf,hf->nh",
        features,
        weights
    )

    z = (
        bias
        + np.logaddexp(
            0.0,
            scale
        ) * anchor
        + gamma * bound
        * np.tanh(
            correction / bound
        )
    )

    pe = expit(z[:, 0])
    pk = expit(z[:, 1])

    predicted = np.column_stack((
        pe,
        (1 - pe) * pk,
        (1 - pe) * (1 - pk)
    ))

    archived = p14[
        (p14.held_cohort == held)
        & (p14.model == "q_residual")
    ].sort_values("gsm")

    aligned = (
        len(archived) == len(a)
        and np.array_equal(
            archived.gsm.to_numpy(),
            a.gsm.to_numpy()
        )
    )

    if aligned:

        err = float(
            np.max(
                np.abs(
                    predicted
                    - archived[PCOLS].to_numpy(float)
                )
            )
        )

        aligned &= np.array_equal(
            archived.true_class.to_numpy(int),
            a.true_class.to_numpy(int)
        )

    else:
        err = float("inf")

    audit(
        f"OUTER_{held}_LABEL_FREE_QUANTILE_CHECKPOINT_REPLAY",
        aligned and err <= 1e-8,
        (
            f"n={len(a)}, frozen residual parameters, "
            "target-only empirical rank"
        ),
        err
    )

    outer_rows.append({
        "outer_held": held,
        "target_n": len(a),
        "max_probability_error": err,
        "guard_lambda": gamma,
        "residual_bound": bound
    })


    # --------------------------------------------------------
    # 12. Compare Phase14 saved channel diagnostics
    # --------------------------------------------------------

    channels = (
        "EGFR128",
        "EGFR512",
        "KRAS128",
        "KRAS512"
    )

    for j, channel in enumerate(channels):

        row = qdiag[
            (qdiag.held_cohort == held)
            & (qdiag.channel == channel)
        ]

        if len(row) != 1:

            audit(
                f"OUTER_DIAGNOSTIC_{held}_{channel}",
                False,
                (
                    "Expected one diagnostic row; "
                    f"found {len(row)}"
                )
            )

            continue

        original = row.iloc[0]

        values = {
            "outer_logit_median": float(
                np.median(logits[:, j])
            ),
            "outer_logit_iqr": float(
                np.percentile(logits[:, j], 75)
                - np.percentile(logits[:, j], 25)
            ),
            "outer_score_ties_fraction": float(
                1 - len(
                    np.unique(
                        logits[:, j]
                    )
                ) / len(logits)
            )
        }

        diff = max(
            abs(
                values[k]
                - float(original[k])
            )
            for k in values
        )

        audit(
            f"OUTER_DIAGNOSTIC_{held}_{channel}",
            diff <= 1e-8,
            (
                "Original Phase14 outer-score "
                "diagnostic replay"
            ),
            diff
        )


# ------------------------------------------------------------
# 13. Preserve unresolved historical limitations
# ------------------------------------------------------------

limitations = src_checks[
    src_checks.status == "WARNING"
][
    ["check", "detail"]
].copy()

extra = pd.DataFrame([
    {
        "check": "HISTORICAL_DEVELOPMENT_EXPOSURE",
        "detail": (
            "Phase12 protocol explicitly records historically "
            "exposed development outcomes."
        )
    },
    {
        "check": "EXTERNAL_METADATA_EXPOSURE",
        "detail": (
            "Phase23 lock hashes verified in Step01B; "
            "hash agreement cannot certify prospective "
            "genotype blindness."
        )
    }
])

limitations = pd.concat(
    [limitations, extra],
    ignore_index=True
)

audit(
    "PHASE12_HISTORICAL_EXPOSURE_DISCLOSED",
    protocol.get(
        "historical_development_outcomes_exposed"
    ) is True,
    (
        "The original protocol flags development "
        "outcomes as historically exposed"
    )
)

audit(
    "PHASE12_SOURCE_HOLDOUT_PROTOCOL",
    (
        "other two outer training cohorts"
        in str(
            protocol.get(
                "oof_strategy",
                ""
            )
        ).lower()
    ),
    protocol.get(
        "oof_strategy",
        ""
    )
)


# ------------------------------------------------------------
# 14. Save results in a NEW revision audit directory
# ------------------------------------------------------------

tag = (
    datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%SZ"
    )
    + "_"
    + uuid.uuid4().hex[:8]
)

OUT = (
    ROOT
    / "99_CoRDiB_REVISION_AUDIT"
    / ("STEP01D_" + tag)
)

OUT.mkdir(
    parents=True,
    exist_ok=False
)

pd.DataFrame(checks).to_csv(
    OUT / "VERIFICATION_CHECKS.csv",
    index=False
)

pd.DataFrame(part_details).to_csv(
    OUT / "VERIFIED_INNER_PARTITIONS.csv",
    index=False
)

pd.DataFrame(inner_rows).to_csv(
    OUT / "OOF_SOURCE_QUANTILE_REPLAY.csv",
    index=False
)

pd.DataFrame(outer_rows).to_csv(
    OUT / "OUTER_FROZEN_RESIDUAL_REPLAY.csv",
    index=False
)

limitations.to_csv(
    OUT / "UNRESOLVED_HISTORICAL_LIMITATIONS.csv",
    index=False
)


# ------------------------------------------------------------
# 15. Final verification status
# ------------------------------------------------------------

fails = [
    x
    for x in checks
    if x["status"] == "FAIL"
]

status = (
    "REVIEW_REQUIRED"
    if fails
    else "VERIFIED_WITH_HISTORICAL_LIMITATIONS"
)

report = {
    "status": status,
    "check_counts": dict(
        Counter(
            x["status"]
            for x in checks
        )
    ),
    "failed_checks": fails,
    "historical_warnings": (
        limitations.to_dict(
            orient="records"
        )
    ),
    "nested_partitions": len(part_details),
    "replayed_inner_channels_from_saved_audit": (
        len(inner_replay)
    ),
    "outer_residual_replays": len(outer_rows),
    "model_fit_or_retraining": False,
    "original_files_modified": False,
    "scope": (
        "Frozen artifact/quantile replays; historical "
        "model-selection independence cannot be certified"
    ),
    "output": str(OUT)
}

(OUT / "SUMMARY.json").write_text(
    json.dumps(
        report,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# 16. Print final audit report
# ------------------------------------------------------------

print("\n" + "=" * 72)
print(
    "CoRDiB STEP 01D - NESTED LOCO "
    "& LEAKAGE EVIDENCE AUDIT"
)
print("=" * 72)

print(
    "ARCHIVED_SOURCE_AUDIT:",
    dict(src_counts)
)

print(
    "ARCHIVED_INNER_OOF_REPLAY:",
    dict(inner_counts)
)

print(
    "NESTED_PARTITIONS_VERIFIED:",
    sum(
        x["partition_valid"]
        for x in part_details
    ),
    "/12"
)

print(
    "INNER_CHANNELS_REPLAY_VERIFIED:",
    int(
        (
            inner_replay.status == "PASS"
        ).sum()
    ),
    "/48"
)

print(
    "MAX_HISTORICAL_INNER_LOGIT_ERROR:",
    f"{inner_replay.max_abs_logit_error.max():.3e}"
)

print(
    "\nOUTER_TARGET_QUANTILE_PREDICTION_REPLAY:"
)

print(
    pd.DataFrame(outer_rows).to_string(
        index=False
    )
)

print(
    "\nAUDIT_CHECKS:",
    dict(
        Counter(
            x["status"]
            for x in checks
        )
    )
)

print("\nUNRESOLVED_LIMITATIONS:")

print(
    limitations.to_string(
        index=False
    )
)

print(
    "\nFAILED_CHECKS:",
    (
        [x["check"] for x in fails]
        if fails
        else "NONE"
    )
)

print("\nSTEP01D_STATUS:", status)
print("AUDIT_OUTPUT:", OUT)
print("MODELS_RETRAINED: FALSE")
print("ORIGINAL_RESEARCH_FILES_MODIFIED: FALSE")


CoRDiB STEP 01D - NESTED LOCO & LEAKAGE EVIDENCE AUDIT
ARCHIVED_SOURCE_AUDIT: {'PASS': 45, 'WARNING': 4}
ARCHIVED_INNER_OOF_REPLAY: {'PASS': 48}
NESTED_PARTITIONS_VERIFIED: 12 /12
INNER_CHANNELS_REPLAY_VERIFIED: 48 /48
MAX_HISTORICAL_INNER_LOGIT_ERROR: 1.144e-05

OUTER_TARGET_QUANTILE_PREDICTION_REPLAY:
outer_held  target_n  max_probability_error  guard_lambda  residual_bound
  GSE13213       117           1.665335e-16           1.0             1.5
  GSE31210       226           2.220446e-16           1.0             1.5
  GSE32863        58           2.220446e-16           1.0             1.5
  GSE72094       442           2.220446e-16           1.0             1.5

AUDIT_CHECKS: {'PASS': 46}

UNRESOLVED_LIMITATIONS:
                           check                                                                                                                                                        detail
CROSS_STUDY_PATIENT_INDEPENDENCE                                                

In [5]:

# ============================================================
# CoRDiB STEP 01E - HISTORICAL DESIGN & INFERENCE EVIDENCE
# Read-only audit of archived research files (Windows Jupyter)
# No training, no refitting, no tuning, no data modification.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import hashlib
import json
import uuid
import numpy as np
import pandas as pd

ROOT = Path(r"D:\LUAD_DATAFIRST_METHOD_2026")
P12 = ROOT / "12_multiresolution_ood_evidence" / "20261023T151555Z_638a3c88"
P14 = ROOT / "14_quantile_residual_transfer" / "20261023T160742Z_f9435239"
P16 = ROOT / "16_source_referenced_residual" / "20261023T170912Z_6ddabbc0"
P17 = ROOT / "17_frozen_reference_inference" / "20261023T174703Z_b0b89644"
P28 = ROOT / "28_matched_mechanism_comparison" / "20261024T062048Z_81cc728e"
STEP01D = (ROOT / "99_CoRDiB_REVISION_AUDIT" /
           "STEP01D_20261029T053604Z_2d8634c0")

COHORT_SIZE = {
    "GSE13213": 117,
    "GSE31210": 226,
    "GSE32863": 58,
    "GSE72094": 442
}

PCOLS = ["p_egfr", "p_kras", "p_dwt"]

required = [
    STEP01D / "SUMMARY.json",
    P12 / "RUN_MANIFEST.json",
    P12 / "PROTOCOL.json",
    P14 / "PROTOCOL.json",
    P14 / "LOCO_PREDICTIONS.csv",
    P16 / "PROTOCOL.json",
    P16 / "LOCO_METRICS.csv",
    P16 / "LOCO_PREDICTIONS.csv",
    P16 / "EQUAL_COHORT_SUMMARY.csv",
    P16 / "SCREENING.json",
    P17 / "PROTOCOL.json",
    P17 / "BANK_EVALUATION.csv",
    P17 / "BANK_PREDICTIONS.csv",
    P17 / "REFERENCE_BANK_MEMBERSHIPS.json",
    P17 / "RANDOM_BANK_PER_COHORT.csv",
    P17 / "EQUAL_COHORT_BANK_SUMMARY.csv",
    P17 / "INVARIANCE_CHECK.json",
    P17 / "SCREENING.json",
    P28 / "AUDIT_PROTOCOL.json"
]

missing = [str(p) for p in required if not p.is_file()]

if missing:
    raise FileNotFoundError(
        "Required original evidence missing:\n"
        + "\n".join(missing)
    )


def read_json(p):
    return json.loads(
        p.read_text(encoding="utf-8-sig")
    )


def sha256(p):
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(
            lambda: f.read(4 * 1024**2), b""
        ):
            h.update(chunk)
    return h.hexdigest()


prior = read_json(STEP01D / "SUMMARY.json")

if prior.get("status") != "VERIFIED_WITH_HISTORICAL_LIMITATIONS":
    raise RuntimeError(
        "STEP01D has not passed: do not proceed."
    )

checks, warnings = [], []


def check(name, ok, detail="", error=None):
    checks.append({
        "check": name,
        "status": "PASS" if bool(ok) else "FAIL",
        "detail": str(detail),
        "max_abs_error": (
            None if error is None else float(error)
        )
    })


def warn(name, detail):
    warnings.append({
        "issue": name,
        "evidence_or_limitation": str(detail)
    })


# ------------------------------------------------------------
# 1. Resolve original capacity-selection parent by SHA256
# ------------------------------------------------------------

p12_manifest = read_json(P12 / "RUN_MANIFEST.json")
p12_protocol = read_json(P12 / "PROTOCOL.json")
p14_protocol = read_json(P14 / "PROTOCOL.json")
p16_protocol = read_json(P16 / "PROTOCOL.json")
p17_protocol = read_json(P17 / "PROTOCOL.json")
p28_protocol = read_json(P28 / "AUDIT_PROTOCOL.json")

parent_hash = p12_manifest.get("parent_manifest_sha256")

parent_matches = [
    p for p in
    (ROOT / "11_representation_capacity_gate").glob(
        "*/RUN_MANIFEST.json"
    )
    if parent_hash and sha256(p) == parent_hash
]

check(
    "PHASE11_PARENT_MANIFEST_LINK",
    len(parent_matches) == 1,
    f"SHA256-matched Phase11 parent runs: {len(parent_matches)}"
)

p11_protocol = None

if len(parent_matches) == 1:

    p11_file = parent_matches[0].parent / "PROTOCOL.json"

    if p11_file.is_file():
        p11_protocol = read_json(p11_file)

    check(
        "PHASE11_CAPACITY_HISTORY",
        bool(p11_protocol)
        and p11_protocol.get("per_head_gene_counts")
        == [24, 128, 512]
        and p11_protocol.get(
            "historical_development_exposure"
        ) is True,
        "24/128/512 exploration with historical outcome exposure"
    )

else:
    warn(
        "PHASE11_PARENT_UNRESOLVED",
        "Cannot uniquely link Phase11 to Phase12"
    )

check(
    "PHASE12_RESOLUTIONS_AND_EXPOSURE",
    p12_protocol.get("resolutions") == [128, 512]
    and p12_protocol.get(
        "historical_development_outcomes_exposed"
    ) is True,
    "Original 128/512 design; development outcomes historically exposed"
)

check(
    "PHASE14_FROZEN_METHOD_PROTOCOL",
    p14_protocol.get("seed") == 20261023
    and p14_protocol.get("epochs") == 400
    and np.isclose(
        float(p14_protocol.get("learning_rate", -1)),
        0.045
    )
    and np.isclose(
        float(p14_protocol.get("correction_cap_per_head", -1)),
        1.5
    )
    and np.isclose(
        float(p14_protocol.get("coefficient_l2", -1)),
        0.02
    ),
    "Recorded seed, epochs, learning rate, bound and residual penalty"
)

check(
    "PHASE28_POSTHOC_COMPARISON_PROTOCOL",
    p28_protocol.get("external_test_used") is False
    and "unchanged" in str(
        p28_protocol.get("original_cordib_model", "")
    ).lower()
    and "exploratory" in str(
        p28_protocol.get("analysis", "")
    ).lower(),
    "Controls compared against archived unchanged Phase14 q_residual"
)


# ------------------------------------------------------------
# 2. Independently recalculate Phase16 equal-cohort summary
# ------------------------------------------------------------

models16 = [
    "raw128",
    "q_anchor",
    "q_static",
    "q_residual",
    "ref0_residual",
    "ref50_residual",
    "ref50_consistency"
]

metrics16 = pd.read_csv(
    P16 / "LOCO_METRICS.csv"
)

summary16 = pd.read_csv(
    P16 / "EQUAL_COHORT_SUMMARY.csv"
)

preds16 = pd.read_csv(
    P16 / "LOCO_PREDICTIONS.csv",
    dtype={"gsm": str}
)

preds14 = pd.read_csv(
    P14 / "LOCO_PREDICTIONS.csv",
    dtype={"gsm": str}
)

identity_counts = metrics16.groupby(
    "model"
)["held_cohort"].nunique()

check(
    "PHASE16_ALL_MODEL_COHORTS",
    len(metrics16) == 28
    and set(identity_counts.index) == set(models16)
    and identity_counts.eq(4).all()
    and not metrics16.duplicated(
        ["model", "held_cohort"]
    ).any(),
    "7 models x 4 development cohorts"
)

metric_cols = [
    "nll",
    "macro_f1",
    "egfr_auc",
    "kras_auc",
    "conditional_kras_nll",
    "brier"
]

recalc = (
    metrics16.groupby(
        "model",
        as_index=False
    )[metric_cols]
    .mean()
)

paired = recalc.merge(
    summary16,
    on="model",
    validate="one_to_one",
    suffixes=("_recalc", "_archived")
)

summary_error = max(
    float(
        np.max(
            np.abs(
                paired[f"{c}_recalc"]
                - paired[f"{c}_archived"]
            )
        )
    )
    for c in metric_cols
)

check(
    "PHASE16_EQUAL_COHORT_AGGREGATION",
    len(paired) == len(models16)
    and summary_error < 1e-10,
    "Independent equal-cohort averaging",
    summary_error
)

for d in (preds16, preds14):

    d["gsm"] = (
        d["gsm"].astype(str).str.strip().str.upper()
    )

    d["held_cohort"] = (
        d["held_cohort"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

p16q = preds16.loc[
    preds16.model == "q_residual",
    ["held_cohort", "gsm", "true_class"] + PCOLS
]

p14q = preds14.loc[
    preds14.model == "q_residual",
    ["held_cohort", "gsm", "true_class"] + PCOLS
]

replayed = p16q.merge(
    p14q,
    how="outer",
    on=["held_cohort", "gsm"],
    suffixes=("_16", "_14"),
    indicator=True,
    validate="one_to_one"
)

matched = (
    len(replayed) == 843
    and replayed["_merge"].eq("both").all()
    and replayed["true_class_16"].eq(
        replayed["true_class_14"]
    ).all()
)

prob_error = (
    max(
        float(
            np.max(
                np.abs(
                    replayed[f"{p}_16"]
                    - replayed[f"{p}_14"]
                )
            )
        )
        for p in PCOLS
    )
    if matched else float("inf")
)

check(
    "PHASE16_USES_FROZEN_PHASE14_CORDIB",
    matched and prob_error <= 1e-10,
    "843 frozen predictions preserved without changes",
    prob_error
)

phase16_screen = read_json(
    P16 / "SCREENING.json"
)

phase16_results = phase16_screen.get(
    "results", {}
)

check(
    "PHASE16_RECORDED_ALTERNATIVE_SCREEN",
    len(phase16_results) == 4
    and all(
        v is False
        for v in phase16_results.values()
    ),
    "All 4 exploratory candidate improvement screens returned False"
)


# ------------------------------------------------------------
# 3. Verify Phase17 reference-bank evidence and summaries
# ------------------------------------------------------------

bank = pd.read_csv(
    P17 / "BANK_EVALUATION.csv"
)

query = pd.read_csv(
    P17 / "BANK_PREDICTIONS.csv",
    dtype={"gsm": str}
)

bank_membership = read_json(
    P17 / "REFERENCE_BANK_MEMBERSHIPS.json"
)

archived_per_cohort = pd.read_csv(
    P17 / "RANDOM_BANK_PER_COHORT.csv"
)

archived_equal = pd.read_csv(
    P17 / "EQUAL_COHORT_BANK_SUMMARY.csv"
)

inv = read_json(
    P17 / "INVARIANCE_CHECK.json"
)

screen17 = read_json(
    P17 / "SCREENING.json"
)

check(
    "PHASE17_REFERENCE_DESIGN",
    p17_protocol.get("not_inductive") is True
    and p17_protocol.get("no_training") is True
    and p17_protocol.get("repeats") == 20
    and "target-reference" in str(
        p17_protocol.get("method", "")
    ).lower(),
    "Frozen bank sampled from held-out target cohort, not training domains"
)

expected_ref = {
    c: {25, n // 2, (3 * n) // 4}
    for c, n in COHORT_SIZE.items()
}

expected_scenarios = 4 * (3 * 20 + 4)

check(
    "PHASE17_SCENARIO_COVERAGE",
    len(bank_membership) == expected_scenarios
    and len(bank) == expected_scenarios * 3
    and set(bank.model) == {
        "q_anchor", "q_static", "q_residual"
    },
    "4 cohorts x [60 random + 4 score-selected] banks x 3 heads"
)

query["gsm"] = (
    query["gsm"]
    .astype(str)
    .str.strip()
    .str.upper()
)

query_groups = {
    (
        str(c), str(s), int(r), str(m)
    ): set(g["gsm"])
    for (c, s, r, m), g in query.groupby(
        ["held_cohort", "scenario", "repeat", "model"],
        sort=False
    )
}

separation_ok, bad = True, []

for item in bank_membership:

    c = str(item["held_cohort"])
    s = str(item["scenario"])
    rep = int(item["repeat"])

    ref = {
        str(g).strip().upper()
        for g in item["reference_gsms"]
    }

    n_ref = int(item["n_reference"])
    n_query = int(item["n_query"])

    good = (
        c in COHORT_SIZE
        and len(ref) == n_ref
        and n_ref + n_query == COHORT_SIZE[c]
        and (
            not s.startswith("random_n")
            or n_ref in expected_ref[c]
        )
    )

    for m in (
        "q_anchor",
        "q_static",
        "q_residual"
    ):

        q = query_groups.get(
            (c, s, rep, m),
            set()
        )

        good &= (
            len(q) == n_query
            and not (q & ref)
        )

    if not good:
        separation_ok = False
        bad.append((c, s, rep))

check(
    "PHASE17_REFERENCE_QUERY_SEPARATION",
    separation_ok
    and len(query_groups) == expected_scenarios * 3,
    f"Source GSM membership and no reference/query overlap; invalid={bad[:3]}"
)

bank_error = max(
    float(
        np.max(
            np.abs(
                bank["nll"]
                - bank["full_q_residual_nll_on_same_queries"]
                - bank["nll_excess_over_full_residual"]
            )
        )
    ),
    float(
        np.max(
            np.abs(
                bank["macro_f1"]
                - bank["full_q_residual_f1_on_same_queries"]
                - bank["f1_change_vs_full_residual"]
            )
        )
    )
)

check(
    "PHASE17_PER_SCENARIO_METRIC_ARITHMETIC",
    bank_error < 1e-10,
    "Recomputed NLL/F1 differences",
    bank_error
)

cols4 = [
    "nll_excess_over_full_residual",
    "f1_change_vs_full_residual",
    "mean_abs_probability_delta_vs_full",
    "changed_class_count_vs_full"
]

random_bank = bank[
    bank.scenario.astype(str).str.startswith("random_n")
]

per_cohort = (
    random_bank.groupby(
        ["held_cohort", "n_reference", "model"],
        as_index=False
    )[cols4]
    .mean()
)


def table_difference(computed, archived, keys, numbers):

    if (
        computed.duplicated(keys).any()
        or archived.duplicated(keys).any()
        or len(computed) != len(archived)
    ):
        return float("inf")

    merged = computed.merge(
        archived,
        how="outer",
        on=keys,
        validate="one_to_one",
        indicator=True,
        suffixes=("_calculated", "_archived")
    )

    if (
        len(merged) != len(computed)
        or not merged["_merge"].eq("both").all()
    ):
        return float("inf")

    return max(
        float(
            np.max(
                np.abs(
                    merged[
                        f"{col}_calculated"
                    ].to_numpy(float)
                    -
                    merged[
                        f"{col}_archived"
                    ].to_numpy(float)
                )
            )
        )
        for col in numbers
    )


err_random = table_difference(
    per_cohort,
    archived_per_cohort,
    ["held_cohort", "n_reference", "model"],
    cols4
)

check(
    "PHASE17_RANDOM_BANK_PER_COHORT_REAGGREGATION",
    err_random < 1e-10,
    "Four-cohort reference-size specific reaggregation",
    err_random
)


def ref_level(r):

    c = str(r["held_cohort"])
    n = int(r["n_reference"])

    if n == 25:
        return "n25"

    if n == COHORT_SIZE[c] // 2:
        return "half"

    if n == (3 * COHORT_SIZE[c]) // 4:
        return "three_quarters"

    raise ValueError(
        f"Unexpected frozen-bank size: {c} {n}"
    )


per_cohort["reference_level"] = (
    per_cohort.apply(
        ref_level,
        axis=1
    )
)

cols3 = cols4[:3]

equal = (
    per_cohort.groupby(
        ["reference_level", "model"],
        as_index=False
    )[cols3]
    .mean()
)

err_equal = table_difference(
    equal,
    archived_equal,
    ["reference_level", "model"],
    cols3
)

check(
    "PHASE17_EQUAL_COHORT_REFERENCE_REAGGREGATION",
    err_equal < 1e-10,
    "Unweighted means over four cohorts",
    err_equal
)

invariance_error = float(
    inv.get(
        "max_query_partition_feature_error",
        float("inf")
    )
)

check(
    "PHASE17_SAVED_FROZEN_BANK_INVARIANCE",
    np.isfinite(invariance_error)
    and invariance_error < 1e-12,
    "Saved held-out-target reference-bank invariance; not prospective inference",
    invariance_error
)


# ------------------------------------------------------------
# 4. Preserve limitations and implementation versions
# ------------------------------------------------------------

check(
    "P14_SOFTWARE_VERSION_RECORD_EXISTS",
    all(
        p14_protocol.get(
            "versions", {}
        ).get(k)
        for k in (
            "python", "numpy", "pandas",
            "scipy", "scikit_learn", "torch"
        )
    ),
    "Historical version strings exist in Phase14 protocol"
)

warn(
    "HISTORICAL_SELECTION_EXPOSURE",
    "Phase11 explored 24/128/512 using historically exposed "
    "cohorts; Phase12+14 used 128/512. Prospective "
    "selection chronology not certified."
)

warn(
    "ORIGINAL_EXECUTABLE_SOURCE",
    "00_DATA_FIRST.ipynb has 0 code cells; HTML exports "
    "contain code, but recorded original fit-call "
    "provenance is not complete."
)

warn(
    "SOURCE_ONLY_REFERENCE_PERFORMANCE",
    "Phase16 ref0_residual is a separately fitted "
    "source-CDF model, not a single-switch ablation "
    "of frozen q_residual coefficients."
)

warn(
    "TARGET_REFERENCE_DEPLOYMENT",
    "Phase17 banks are built from each held-out target "
    "cohort; batch invariance with a frozen bank is not "
    "proof of inductive single-patient validity."
)

warn(
    "EXTERNAL_LABEL_EXPOSURE",
    "GSE66863 labels were historically accessible before "
    "prediction locking; checksum matches cannot "
    "certify prospective blinding."
)

warn(
    "PATIENT_INDEPENDENCE",
    "Patient-level nonoverlap across source studies "
    "remains unverified."
)


# ------------------------------------------------------------
# 5. Save independent ledgers into a new audit directory
# ------------------------------------------------------------

tag = (
    datetime.now(timezone.utc).strftime(
        "%Y%m%dT%H%M%SZ"
    )
    + "_"
    + uuid.uuid4().hex[:8]
)

OUT = (
    ROOT
    / "99_CoRDiB_REVISION_AUDIT"
    / ("STEP01E_" + tag)
)

OUT.mkdir(
    parents=True,
    exist_ok=False
)

pd.DataFrame(checks).to_csv(
    OUT / "VERIFICATION_CHECKS.csv",
    index=False
)

pd.DataFrame(warnings).to_csv(
    OUT / "HISTORICAL_LIMITATIONS.csv",
    index=False
)

recalc.to_csv(
    OUT / "RECALCULATED_SOURCE_REFERENCE_METRICS.csv",
    index=False
)

equal.to_csv(
    OUT / "RECALCULATED_FROZEN_BANK_SUMMARY.csv",
    index=False
)

settings = {
    "phase11_capacity": (
        p11_protocol or {}
    ).get("per_head_gene_counts"),

    "phase11_historically_exposed": (
        p11_protocol or {}
    ).get("historical_development_exposure"),

    "phase12_resolutions": p12_protocol.get(
        "resolutions"
    ),

    "phase12_seed": p12_protocol.get(
        "seed"
    ),

    "phase12_historical_exposure": p12_protocol.get(
        "historical_development_outcomes_exposed"
    ),

    "phase14_seed": p14_protocol.get(
        "seed"
    ),

    "phase14_epochs": p14_protocol.get(
        "epochs"
    ),

    "phase14_learning_rate": p14_protocol.get(
        "learning_rate"
    ),

    "phase14_correction_bound": p14_protocol.get(
        "correction_cap_per_head"
    ),

    "phase14_coefficient_l2": p14_protocol.get(
        "coefficient_l2"
    ),

    "phase14_group_penalty": p14_protocol.get(
        "group_penalty"
    ),

    "phase14_historical_software": p14_protocol.get(
        "versions", {}
    ),

    "phase16_reference": p16_protocol.get(
        "reference"
    ),

    "phase16_models": p16_protocol.get(
        "models", {}
    ),

    "phase17_reference_protocol": p17_protocol.get(
        "method"
    ),

    "phase17_not_inductive": p17_protocol.get(
        "not_inductive"
    ),

    "phase28_comparator_design": p28_protocol.get(
        "controls"
    ),

    "base_logistic_details_from_HTML_not_confirmed_by_fit_logs": {
        "C": 0.2,
        "solver": "lbfgs",
        "max_iter": 400,
        "status": (
            "Read from supplied Phase12 HTML source; "
            "original code/fit logs unavailable"
        )
    }
}

(OUT / "IMPLEMENTATION_EVIDENCE.json").write_text(
    json.dumps(
        settings,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

failures = [
    x for x in checks
    if x["status"] == "FAIL"
]

status = (
    "REVIEW_REQUIRED"
    if failures
    else "ARCHIVED_DESIGN_VERIFIED_WITH_LIMITATIONS"
)

report = {
    "status": status,
    "checks": dict(
        Counter(
            x["status"]
            for x in checks
        )
    ),
    "failures": failures,
    "limitations": warnings,
    "phase16_model_nll": dict(
        zip(
            recalc.model,
            recalc.nll
        )
    ),
    "phase17_saved_invariance_error": invariance_error,
    "source_alternatives_screen": phase16_results,
    "frozen_bank_screen": screen17,
    "original_research_files_modified": False,
    "model_retrained": False,
    "output": str(OUT)
}

(OUT / "SUMMARY.json").write_text(
    json.dumps(
        report,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# 6. Print consolidated results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print(
    "CoRDiB STEP 01E - DESIGN & INFERENCE EVIDENCE AUDIT"
)
print("=" * 70)

print(
    "P11_PARENT_HASH_MATCHES:",
    len(parent_matches)
)

print("\nPHASE16_MODELS:")

print(
    recalc[
        ["model", "nll", "macro_f1"]
    ].round(6).to_string(index=False)
)

print(
    "\nPHASE16_ALTERNATIVE_SCREENS:",
    phase16_results
)

print("\nPHASE17_Q_RESIDUAL_FROZEN_BANKS:")

print(
    equal.loc[
        equal.model == "q_residual",
        ["reference_level"] + cols3
    ].round(6).to_string(index=False)
)

print(
    "PHASE17_INVARIANCE_ERROR:",
    f"{invariance_error:.3e}"
)

print(
    "\nRECORDED_PHASE14_SOFTWARE_VERSIONS:",
    settings["phase14_historical_software"]
)

print(
    "RECORDED_RESIDUAL_SETTINGS:",
    {
        k: settings[k]
        for k in (
            "phase14_epochs",
            "phase14_learning_rate",
            "phase14_correction_bound",
            "phase14_coefficient_l2"
        )
    }
)

print("\nAUDIT_CHECKS:", report["checks"])

print(
    "FAILED_CHECKS:",
    [
        x["check"]
        for x in failures
    ] or "NONE"
)

print(
    "HISTORICAL_LIMITATION_CATEGORIES:",
    [
        x["issue"]
        for x in warnings
    ]
)

print("\nSTEP01E_STATUS:", status)
print("AUDIT_OUTPUT:", OUT)
print("MODELS_RETRAINED: FALSE")
print("ORIGINAL_RESEARCH_FILES_MODIFIED: FALSE")


CoRDiB STEP 01E - DESIGN & INFERENCE EVIDENCE AUDIT
P11_PARENT_HASH_MATCHES: 1

PHASE16_MODELS:
            model      nll  macro_f1
         q_anchor 0.920973  0.525022
       q_residual 0.909271  0.545245
         q_static 0.913136  0.525672
           raw128 1.137068  0.511660
    ref0_residual 1.400331  0.233139
ref50_consistency 0.942601  0.425182
   ref50_residual 0.938593  0.426325

PHASE16_ALTERNATIVE_SCREENS: {'macro_f1_gain_vs_original_at_least_0_01': False, 'nll_gain_vs_original_at_least_0_01': False, 'macro_f1_gain_vs_matched_plain_at_least_0_01': False, 'random_half_stability_gain_at_least_0_01': False}

PHASE17_Q_RESIDUAL_FROZEN_BANKS:
reference_level  nll_excess_over_full_residual  f1_change_vs_full_residual  mean_abs_probability_delta_vs_full
           half                      -0.000158                   -0.009921                            0.018682
            n25                       0.022200                   -0.025242                            0.041026
 three_q